# Práctica 4 / 4b · Detección de vehículos y matrículas + OCR

| Sección | Contenido |
|---|---|
| 0 | Imports, configuración y utilidades comunes |
| 1 | Dataset de matrículas |
| 2 | Entrenamiento de YOLO y RT-DETR |
| 2b | RT-DETR sin Ultralytics (repo oficial): dataset COCO, entrenamiento y envoltorio |
| 3 | Comparativa YOLO vs RT-DETR |
| 4 | OCR de matrículas (Tesseract / PaddleOCR-VL) |
| 5 | Extras del vídeo: flujo por direcciones y anonimización |
| 6 | Pipeline de vídeo |
| 7 | Comparativa OCR clásico vs VLM |

**Orden de ejecución:** la sección 0 siempre. Las secciones 1 y 2 solo para rehacer el dataset o reentrenar. El pipeline (6) necesita las secciones 4 y 5, y la comparativa OCR (7) necesita la 4. Para usar el RT-DETR sin Ultralytics en la comparativa (3) y en el pipeline (6), ejecuta antes la primera celda de la 2b (su dataset y su entrenamiento, solo una vez).

## 0. Imports, configuración y utilidades

In [2]:
# ---------------- Imports de todo el cuaderno ----------------
import csv
import importlib.util
import json
import os
import platform
import queue
import random
import re
import threading
import time
import warnings
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import count
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from PIL import Image
from ultralytics import RTDETR, YOLO
from ultralytics.data.utils import IMG_FORMATS, check_det_dataset
from ultralytics.utils.plotting import colors
from ultralytics.utils.torch_utils import get_flops

warnings.filterwarnings("ignore", category=UserWarning)

# Dependencias que solo se cargan al usarlas: aquí solo se comprueba que están instaladas
OPCIONALES = {
    "pytesseract": "OCR Tesseract (pip install pytesseract + programa Tesseract)",
    "transformers": "OCR PaddleOCR-VL (pip install \"transformers>=5.0.0\")",
    "psutil": "RAM en las especificaciones del equipo (pip install psutil)",
    "lap": "tracking de Ultralytics (pip install lap)",
    "supervision": "tracking del RT-DETR sin Ultralytics, sección 2b (pip install \"supervision<0.31\")",
    "thop": "GFLOPs del RT-DETR sin Ultralytics en la comparativa (pip install thop)",
}
for modulo, uso in OPCIONALES.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Falta {modulo}: necesario para {uso}")
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")


ultralytics 8.4.190 | torch 2.6.0+cu124 | CUDA: True


In [3]:
# ---------------- Configuración común ----------------
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Dataset de matrículas (sección 1)
CARPETA_DATOS = Path("datos")
CARPETA_IMAGENES = CARPETA_DATOS / "images"
CARPETA_ANOTACIONES = CARPETA_DATOS / "annotations"
DATA = str(CARPETA_DATOS / "data.yaml")

# Modelos de matrículas entrenados (sección 2): comprobar que son los últimos entrenamientos
RUTA_YOLO_MATRICULAS = "runs/detect/train-3/weights/best.pt"
RUTA_RTDETR_MATRICULAS = "runs/detect/rtdetr-2/weights/best.pt"
IMGSZ_YOLO = 640
IMGSZ_RTDETR = 640       # RT-DETR está preentrenado a 640: se entrena y se usa a 640

VIDEO = "C0142.MP4"
CARPETA_RESULTADOS = Path("results")
CARPETA_RESULTADOS.mkdir(exist_ok=True)


In [4]:
# ---------------- Utilidades comunes ----------------
def especificaciones_equipo():
    """CPU, RAM, GPU y versiones: van al README porque los tiempos dependen del equipo."""
    cpu = platform.processor() or platform.machine()
    if platform.system() == "Windows":           # platform.processor() no da el nombre comercial
        try:
            import winreg
            clave = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                   r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            cpu = winreg.QueryValueEx(clave, "ProcessorNameString")[0].strip()
        except OSError:
            pass
    try:
        import psutil
        ram = f"{psutil.virtual_memory().total / 1024 ** 3:.0f} GB"
    except ImportError:
        ram = "?"
    if torch.cuda.is_available():
        propiedades = torch.cuda.get_device_properties(0)
        gpu = f"{propiedades.name} ({propiedades.total_memory / 1024 ** 3:.0f} GB)"
    else:
        gpu = "sin GPU"
    return {"SO": f"{platform.system()} {platform.release()}", "CPU": cpu, "RAM": ram, "GPU": gpu,
            "Python": platform.python_version(), "torch": torch.__version__,
            "CUDA": torch.version.cuda or "-", "ultralytics": ultralytics.__version__}


def tamano_imagen(ruta):
    """(ancho, alto) leyendo solo la cabecera y teniendo en cuenta la orientación EXIF, como cv2.imread y Ultralytics."""
    with Image.open(ruta) as imagen:
        ancho, alto = imagen.size
        if imagen.getexif().get(274) in (5, 6, 7, 8):   # rotada 90/270 grados
            ancho, alto = alto, ancho
    return ancho, alto


def args_entrenamiento(ruta_pesos):
    """Argumentos con los que se entrenó un modelo (quedan guardados dentro del .pt)."""
    return torch.load(ruta_pesos, map_location="cpu", weights_only=False).get("train_args") or {}


def imagenes_del_split(data=DATA, split="val"):
    """Imágenes de un split, resueltas igual que hace Ultralytics (carpeta, .txt o lista)."""
    info = check_det_dataset(data)
    fuentes = info[split] if isinstance(info[split], list) else [info[split]]
    imagenes = []
    for fuente in map(Path, fuentes):
        if fuente.is_dir():
            imagenes += sorted(ruta for ruta in fuente.rglob("*") if ruta.suffix[1:].lower() in IMG_FORMATS)
        else:   # .txt con una ruta por línea, relativas a la carpeta del .txt
            for linea in fuente.read_text().splitlines():
                if linea.strip():
                    ruta = Path(linea.strip())
                    imagenes.append(ruta if ruta.is_absolute() else (fuente.parent / ruta).resolve())
    return imagenes


def cronometrar(funcion, *args, **kwargs):
    """Ejecuta funcion(*args, **kwargs) y devuelve (resultado, milisegundos). Espera a la GPU."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return resultado, (time.perf_counter() - inicio) * 1000


def centro(caja):
    return (caja[0] + caja[2]) / 2, (caja[1] + caja[3]) / 2


def ampliar_caja(caja, margen, ancho_frame, alto_frame):
    """Caja (x1, y1, x2, y2) ampliada un `margen` relativo por lado y ajustada al frame, o None si queda vacía."""
    x1, y1, x2, y2 = map(float, caja)
    margen_x, margen_y = (x2 - x1) * margen, (y2 - y1) * margen
    x1, y1 = max(0, int(x1 - margen_x)), max(0, int(y1 - margen_y))
    x2, y2 = min(ancho_frame, int(x2 + margen_x)), min(alto_frame, int(y2 + margen_y))
    return (x1, y1, x2, y2) if x2 > x1 and y2 > y1 else None


def dibujar_panel(imagen, lineas, derecha=False):
    """Recuadro negro con texto blanco en una esquina superior (legible también en 4K)."""
    if not lineas:
        return
    escala = max(imagen.shape[1] / 1280, 1.0)
    alto_linea, ancho_panel = int(30 * escala), int(330 * escala)
    x0 = imagen.shape[1] - ancho_panel if derecha else 0
    cv2.rectangle(imagen, (x0, 0), (x0 + ancho_panel, alto_linea * len(lineas) + 10), (0, 0, 0), -1)
    for posicion, linea in enumerate(lineas, start=1):
        cv2.putText(imagen, linea, (x0 + 10, alto_linea * posicion), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7 * escala, (255, 255, 255), 2)


def guardar_csv(ruta, cabecera, filas):
    with open(ruta, "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(cabecera)
        escritor.writerows(filas)


def tabla_markdown(tabla, columnas, decimales=3):
    """DataFrame -> tabla Markdown para pegar en el README."""
    lineas = ["| " + (tabla.index.name or "") + " | " + " | ".join(columnas) + " |",
              "|---" * (len(columnas) + 1) + "|"]
    for nombre, fila in tabla[columnas].iterrows():
        valores = [f"{valor:.{decimales}f}" if isinstance(valor, (float, np.floating)) else str(valor)
                   for valor in fila]
        lineas.append(f"| {nombre} | " + " | ".join(valores) + " |")
    return "\n".join(lineas)


def grafica_barras(tabla, columnas, colores, titulo, ruta, formato="{:.3f}", eje_y="", log=False):
    """Barras agrupadas: una serie por fila de `tabla` (con su color) y un grupo por columna."""
    figura, eje = plt.subplots(figsize=(max(6, 2.2 * len(columnas)), 4.5))
    posiciones = np.arange(len(columnas))
    ancho = 0.8 / len(tabla)
    for indice, (nombre, fila) in enumerate(tabla[columnas].iterrows()):
        valores = fila.astype(float).values
        barras = eje.bar(posiciones + (indice - (len(tabla) - 1) / 2) * ancho, valores, ancho,
                         label=nombre, color=colores[indice])
        eje.bar_label(barras, [formato.format(valor) for valor in valores], fontsize=8, padding=2)
    eje.set_xticks(posiciones, columnas)
    if log:
        eje.set_yscale("log")
    eje.set(title=titulo, ylabel=eje_y)
    eje.legend(); eje.grid(axis="y", alpha=0.3)
    figura.tight_layout(); figura.savefig(ruta, dpi=150)
    plt.show()


EQUIPO = especificaciones_equipo()
for clave, valor in EQUIPO.items():
    print(f"{clave:>12}: {valor}")
for ruta in (RUTA_YOLO_MATRICULAS, RUTA_RTDETR_MATRICULAS):
    if Path(ruta).exists():
        args = args_entrenamiento(ruta)
        print(f"{ruta}: imgsz={args.get('imgsz')}, epochs={args.get('epochs')}, base={args.get('model')}")
    else:
        print(f"NO EXISTE: {ruta} -> revisa la carpeta en runs/detect/ (o entrena en la sección 2)")


          SO: Windows 10
         CPU: AMD Ryzen 7 7735HS with Radeon Graphics
         RAM: 31 GB
         GPU: NVIDIA GeForce RTX 4060 Laptop GPU (8 GB)
      Python: 3.11.18
       torch: 2.6.0+cu124
        CUDA: 12.4
 ultralytics: 8.4.190
runs/detect/train-3/weights/best.pt: imgsz=640, epochs=100, base=yolo11n.pt
runs/detect/rtdetr-2/weights/best.pt: imgsz=640, epochs=100, base=rtdetr-l.pt


## 1. Dataset de matrículas

Convierte las anotaciones de `datos/annotations/*.json` a etiquetas YOLO y genera `train.txt`, `val.txt` y `data.yaml` (80/20, semilla 42).

In [ ]:
# Convierte las anotaciones JSON a formato YOLO y reparte train/val 80/20 (solo hace falta una vez)
CARPETA_LABELS = CARPETA_DATOS / "labels"
PROPORCION_VAL = 0.2
CARPETA_LABELS.mkdir(exist_ok=True)

imagenes = []                       # solo las que tienen anotación
sin_json = 0
for imagen in sorted(ruta for ruta in CARPETA_IMAGENES.iterdir() if ruta.suffix.lower() == ".jpg"):
    ruta_json = CARPETA_ANOTACIONES / f"{imagen.stem}.json"
    if not ruta_json.exists():
        sin_json += 1
        continue
    lineas = []
    for objeto in json.loads(ruta_json.read_text()):
        centro_x, centro_y, ancho, alto = objeto["rectangle"]   # centro y tamaño, normalizados
        lineas.append(f"0 {centro_x:.6f} {centro_y:.6f} {ancho:.6f} {alto:.6f}")
    (CARPETA_LABELS / f"{imagen.stem}.txt").write_text("\n".join(lineas))
    imagenes.append(imagen)

random.seed(42)
random.shuffle(imagenes)
numero_val = int(len(imagenes) * PROPORCION_VAL)
val, train = imagenes[:numero_val], imagenes[numero_val:]

(CARPETA_DATOS / "train.txt").write_text("\n".join(f"./images/{imagen.name}" for imagen in train))
(CARPETA_DATOS / "val.txt").write_text("\n".join(f"./images/{imagen.name}" for imagen in val))
Path(DATA).write_text(f"path: {CARPETA_DATOS.resolve()}\ntrain: train.txt\nval: val.txt\nnames:\n  0: plate\n")
print(f"Listo: {len(train)} train / {len(val)} val | sin JSON: {sin_json}")


## 2. Entrenamiento

In [ ]:
# Entrenamiento de YOLO para matrículas (solo una vez). Solo se indican los valores distintos de los de Ultralytics.
modelo = YOLO("yolo11n.pt")
modelo.train(
    data=DATA, epochs=100, imgsz=IMGSZ_YOLO, batch=16, seed=42,
    hsv_s=0.6, hsv_v=0.5,      # más variación de saturación y brillo (día, noche, sombras)
    degrees=5.0, shear=2.0,    # cámaras algo inclinadas
    perspective=0.0005,        # perspectiva leve
    fliplr=0.0,                # sin espejo: el texto de la matrícula no debe salir invertido
    mixup=0.1,                 # mezcla suave de dos imágenes
)
# Al terminar, pon en RUTA_YOLO_MATRICULAS la carpeta que indique la salida


In [ ]:
# Entrenamiento de RT-DETR para matrículas (solo una vez)
modelo = RTDETR("rtdetr-l.pt")   # preentrenado en COCO; la cabeza se reajusta a 1 clase (plate)
modelo.train(
    data=DATA, epochs=100, imgsz=IMGSZ_RTDETR, batch=8, device=DEVICE, cache=True,
    patience=30,                   # los DETR convergen más despacio que YOLO
    optimizer="AdamW", lr0=0.0001, # optimizador y lr habituales en modelos tipo DETR
    deterministic=False,           # RT-DETR usa operaciones sin versión determinista en CUDA
    name="rtdetr",
)
# Al terminar, pon en RUTA_RTDETR_MATRICULAS la carpeta que indique la salida


## 2b. RT-DETR sin Ultralytics (repo oficial)

RT-DETRv2-L (ResNet-50) del repositorio oficial `lyuwenyu/RT-DETR` (carpeta `RT-DETR/`), cargado como en `load_rtdetr.py` y `VC_P4_RTDETR.ipynb`. Dos modelos:

- **General** (personas y vehículos): los pesos oficiales de COCO, sin reentrenar.
- **Matrículas**: el mismo modelo reentrenado con el solver del repo sobre el reparto de la sección 1, con un *data augmentation* equivalente al de YOLO (color, rotación, shear, perspectiva, escala y traslación; sin volteo, sin mosaic y sin mixup).

`RTDETRNativo` les da la interfaz que usa el resto del cuaderno (`.names` y `.track()`), así que la comparativa (3) y el pipeline (6) los usan igual que a los de Ultralytics. El tracking lo hace `sv.ByteTrack` (`make_tracker`) con los valores de `tracker_general.yaml` y `tracker_matriculas.yaml` (umbral, buffer y `match_thresh`; ByteTrack no tiene ReID ni compensación de cámara como el BoT-SORT general).

Las métricas de la comparativa se calculan con las mismas funciones que `model.val()` de Ultralytics, para que las tres filas sean comparables.

In [11]:
# ---------------- RT-DETR sin Ultralytics: configuración, carga, envoltorio y evaluación ----------------
# Usa el repo oficial (RT-DETR/rtdetrv2_pytorch) y load_rtdetr.py. Ejecutar siempre que se use el RT-DETR nativo.
import copy
import sys
import urllib.request
from types import SimpleNamespace

import supervision as sv
import torchvision.transforms.v2 as T
import yaml
from PIL import ImageOps
from ultralytics.engine.validator import BaseValidator     # solo para medir igual que model.val() (sección 3)
from ultralytics.utils.metrics import ConfusionMatrix, DetMetrics, box_iou

REPO_RTDETR = Path("RT-DETR/rtdetrv2_pytorch")
if str(REPO_RTDETR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_RTDETR.resolve()))

import load_rtdetr as modulo_load_rtdetr
from load_rtdetr import load_rtdetr
from src.core import YAMLConfig, register
from src.core.yaml_utils import load_config
from src.data.dataset import CocoDetection, mscoco_category2name, mscoco_label2category
from src.misc import dist_utils
from src.solver import TASKS

# ---------------- Configuración ----------------
NATIVO_FAMILIA, NATIVO_TAMANO = "v2", "l"     # RT-DETRv2 con ResNet-50, como en el ejemplo (≈ rtdetr-l de Ultralytics)
NATIVO_IMGSZ = 640                             # fija: entrenamiento e inferencia (eval_spatial_size del repo)
NATIVO_COCO = "coco"                           # "pesos" del modelo general: los oficiales de COCO
_cfg_nativo, _tag_nativo, _fichero_nativo = modulo_load_rtdetr.MODELS[NATIVO_FAMILIA][
    modulo_load_rtdetr.SIZE_ALIASES.get(NATIVO_TAMANO, NATIVO_TAMANO)]
NATIVO_CONFIG_BASE = REPO_RTDETR / "configs" / _cfg_nativo
NATIVO_PESOS_COCO = Path("weights") / _fichero_nativo       # misma ruta que usa load_rtdetr
NOMBRES_COCO = {etiqueta: mscoco_category2name[categoria] for etiqueta, categoria in mscoco_label2category.items()}
DISPOSITIVO_NATIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Entrenamiento de matrículas
CARPETA_NATIVO = Path("runs/rtdetr_nativo")
NATIVO_CONFIG_MATRICULAS = CARPETA_NATIVO / "rtdetrv2_matriculas.yml"
RUTA_RTDETR_NATIVO_MATRICULAS = str(CARPETA_NATIVO / "best.pth")
COCO_TRAIN, COCO_VAL = CARPETA_DATOS / "coco_train.json", CARPETA_DATOS / "coco_val.json"
NATIVO_EPOCAS = 100
NATIVO_BATCH = 4                 # bájalo si no cabe en la GPU
NATIVO_SEMILLA = 42
NATIVO_WARMUP_EPOCAS = 3         # como warmup_epochs de Ultralytics

# Data augmentation equivalente al de YOLO (sección 2): sin volteo (el texto saldría invertido), sin mosaic ni mixup
AUMENTO_NATIVO = [
    {"type": "ColorJitter", "brightness": 0.5, "saturation": 0.6, "hue": 0.015},     # hsv_v, hsv_s, hsv_h
    {"type": "RandomAffine", "degrees": 5.0, "translate": [0.1, 0.1], "scale": [0.5, 1.5],
     "shear": [-2.0, 2.0, -2.0, 2.0], "fill": 114},                                  # degrees, translate, scale, shear
    {"type": "RandomPerspective", "distortion_scale": 0.15, "p": 1.0, "fill": 114},   # ≈ perspective=0.0005
    {"type": "SanitizeBoundingBoxes", "min_size": 1},
    {"type": "Resize", "size": [NATIVO_IMGSZ, NATIVO_IMGSZ]},
    {"type": "SanitizeBoundingBoxes", "min_size": 1},
    {"type": "ConvertPILImage", "dtype": "float32", "scale": True},
    {"type": "ConvertBoxes", "fmt": "cxcywh", "normalize": True},
]

# El repo crea las transformaciones por nombre desde su registro: se añaden las de torchvision que no trae
for transformacion in (T.ColorJitter, T.RandomAffine, T.RandomPerspective):
    register(force=True)(transformacion)


@register(force=True)
class CocoDetectionExif(CocoDetection):
    """CocoDetection del repo aplicando la orientación EXIF, como cv2.imread y Ultralytics."""

    def _load_image(self, id):
        with Image.open(os.path.join(self.root, self.coco.loadImgs(id)[0]["file_name"])) as imagen:
            return ImageOps.exif_transpose(imagen).convert("RGB")


# ---------------- Datos ----------------
def etiquetas_yolo(ruta_imagen, ancho, alto):
    """Cajas (x1, y1, x2, y2) en píxeles de datos/labels/<imagen>.txt (las de la sección 1)."""
    ruta = CARPETA_DATOS / "labels" / f"{Path(ruta_imagen).stem}.txt"
    filas = np.loadtxt(ruta, ndmin=2) if ruta.exists() and ruta.stat().st_size else np.zeros((0, 5))
    centro_x, centro_y = filas[:, 1] * ancho, filas[:, 2] * alto
    mitad_ancho, mitad_alto = filas[:, 3] * ancho / 2, filas[:, 4] * alto / 2
    return np.stack([centro_x - mitad_ancho, centro_y - mitad_alto,
                     centro_x + mitad_ancho, centro_y + mitad_alto], axis=1)


def escribir_config_nativo(n_train=None):
    """Config del repo para matrículas: hereda la de COCO y cambia clases, datos, aumento y entrenamiento."""
    CARPETA_NATIVO.mkdir(parents=True, exist_ok=True)
    warmup = max(100, NATIVO_WARMUP_EPOCAS * n_train // NATIVO_BATCH) if n_train else 2000   # en iteraciones

    def datos(anotaciones, operaciones):
        return {"type": "CocoDetectionExif", "img_folder": CARPETA_IMAGENES.resolve().as_posix(),
                "ann_file": anotaciones.resolve().as_posix(), "return_masks": False,
                "transforms": {"type": "Compose", "ops": operaciones, "policy": {"name": "default"}}}

    config = {
        "__include__": [Path(os.path.relpath(NATIVO_CONFIG_BASE, CARPETA_NATIVO)).as_posix()],
        "output_dir": CARPETA_NATIVO.as_posix(),
        "num_classes": 1, "remap_mscoco_category": False,
        "epoches": NATIVO_EPOCAS,
        "checkpoint_freq": 10 ** 6,          # solo last.pth y best.pth (no un .pth por época)
        "print_freq": 5,
        "use_amp": True, "use_ema": True, "sync_bn": False,
        "PResNet": {"pretrained": False},    # el backbone ya viene en los pesos COCO
        "lr_warmup_scheduler": {"warmup_duration": warmup},
        "train_dataloader": {
            "dataset": datos(COCO_TRAIN, AUMENTO_NATIVO),
            "collate_fn": {"type": "BatchImageCollateFunction", "scales": None, "stop_epoch": None},  # sin multiescala
            "total_batch_size": NATIVO_BATCH, "num_workers": 0, "shuffle": True},
        "val_dataloader": {
            "dataset": datos(COCO_VAL, [{"type": "Resize", "size": [NATIVO_IMGSZ, NATIVO_IMGSZ]},
                                        {"type": "ConvertPILImage", "dtype": "float32", "scale": True}]),
            "total_batch_size": NATIVO_BATCH, "num_workers": 0, "shuffle": False},
    }
    NATIVO_CONFIG_MATRICULAS.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")


def asegurar_pesos_coco():
    """Descarga los pesos COCO oficiales si no están (lo mismo que hace load_rtdetr)."""
    if not NATIVO_PESOS_COCO.exists():
        NATIVO_PESOS_COCO.parent.mkdir(exist_ok=True)
        print(f"Descargando {NATIVO_PESOS_COCO}...")
        urllib.request.urlretrieve(
            f"https://github.com/lyuwenyu/storage/releases/download/{_tag_nativo}/{_fichero_nativo}", NATIVO_PESOS_COCO)


# ---------------- Tracking ----------------
def make_tracker(buffer_frames, fps_video, min_conf=0.7, match_thresh=0.7, min_frames=1):
    """sv.ByteTrack: el RT-DETR nativo no trae tracking (min_frames=1: el mínimo ya lo aplica RenumeradorIds)."""
    return sv.ByteTrack(
        track_activation_threshold=min_conf,       # score mínimo para iniciar un track
        lost_track_buffer=int(buffer_frames),      # frames que aguanta un track sin detección
        minimum_matching_threshold=match_thresh,
        frame_rate=int(fps_video),
        minimum_consecutive_frames=min_frames,
    )


# ---------------- Carga ----------------
def cargar_rtdetr_matriculas(ruta, dispositivo=DISPOSITIVO_NATIVO):
    """Modelo y postprocesador de matrículas entrenado en esta sección (como load_rtdetr, con 1 clase)."""
    if not NATIVO_CONFIG_MATRICULAS.exists():
        escribir_config_nativo()
    load_config.__defaults__[0].clear()      # load_config acumula configs en un dict por defecto
    cfg = YAMLConfig(str(NATIVO_CONFIG_MATRICULAS))
    estado = torch.load(ruta, map_location="cpu", weights_only=False)
    cfg.model.load_state_dict(estado["ema"]["module"] if "ema" in estado else estado["model"])
    return cfg.model.deploy().to(dispositivo), cfg.postprocessor.deploy().to(dispositivo)


class _CajasNativas:
    """Imita results[0].boxes de Ultralytics: lo que lee detecciones() del pipeline."""

    def __init__(self, xyxy, conf, cls, ids=None):
        self.xyxy = torch.as_tensor(np.asarray(xyxy, dtype=np.float32)).reshape(-1, 4)
        self.conf = torch.as_tensor(np.asarray(conf, dtype=np.float32)).reshape(-1)
        self.cls = torch.as_tensor(np.asarray(cls, dtype=np.float32)).reshape(-1)
        self.id = None if ids is None else torch.as_tensor(np.asarray(ids, dtype=np.float32)).reshape(-1)

    def __len__(self):
        return len(self.conf)


class RTDETRNativo:
    """RT-DETR del repo oficial con la interfaz que usa el cuaderno: .names y .track(frame, ...)[0].boxes.
    pesos: NATIVO_COCO (modelo general con los pesos oficiales) o la ruta de un checkpoint de matrículas."""

    def __init__(self, pesos=NATIVO_COCO, dispositivo=DISPOSITIVO_NATIVO):
        self.dispositivo = torch.device(dispositivo)
        if pesos == NATIVO_COCO:
            self.modelo, self.postprocesador = load_rtdetr(family=NATIVO_FAMILIA, size=NATIVO_TAMANO,
                                                           device=self.dispositivo)
            self.names = dict(NOMBRES_COCO)         # mismos índices que las clases COCO de YOLO
        else:
            self.modelo, self.postprocesador = cargar_rtdetr_matriculas(pesos, self.dispositivo)
            self.names = {0: "plate"}
        self.fps = 30.0
        self.trackers = {}                          # yaml del tracker -> sv.ByteTrack
        self.speed = {}                             # ms de la última predicción, como result.speed de Ultralytics
        self._avisado = False

    def configurar_video(self, fps):
        """Fps del vídeo para el tracker; empieza con los tracks vacíos."""
        self.fps, self.trackers = float(fps), {}

    def _sincronizar(self):
        if self.dispositivo.type == "cuda":
            torch.cuda.synchronize()

    @torch.no_grad()
    def predecir(self, bgr, conf=0.25, classes=None, fp16=False):
        """(cajas xyxy en píxeles, confianzas, clases) de un frame BGR, de mayor a menor confianza.
        Mismo preprocesado que el ejemplo: RGB, 640x640 sin letterbox y valores en [0, 1]."""
        alto, ancho = bgr.shape[:2]
        inicio = time.perf_counter()
        imagen = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)).resize((NATIVO_IMGSZ, NATIVO_IMGSZ),
                                                                               Image.Resampling.BILINEAR)
        entrada = torch.from_numpy(np.array(imagen)).permute(2, 0, 1).float().div(255)[None].to(self.dispositivo)
        self._sincronizar()
        fin_preproceso = time.perf_counter()
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16 and self.dispositivo.type == "cuda"):
            salida = self.modelo(entrada)
        self._sincronizar()
        fin_inferencia = time.perf_counter()
        salida = {"pred_logits": salida["pred_logits"].float(), "pred_boxes": salida["pred_boxes"].float()}
        etiquetas, cajas, puntuaciones = self.postprocesador(
            salida, torch.tensor([[ancho, alto]], device=self.dispositivo))
        etiquetas, cajas, puntuaciones = etiquetas[0].cpu().numpy(), cajas[0].cpu().numpy(), puntuaciones[0].cpu().numpy()
        mantener = puntuaciones >= conf
        if classes is not None:
            mantener &= np.isin(etiquetas, classes)
        cajas = np.clip(cajas[mantener], 0, [ancho, alto, ancho, alto]).astype(np.float32)
        resultado = cajas, puntuaciones[mantener].astype(np.float32), etiquetas[mantener].astype(int)
        fin = time.perf_counter()
        self.speed = {"preprocess": (fin_preproceso - inicio) * 1000,
                      "inference": (fin_inferencia - fin_preproceso) * 1000,
                      "postprocess": (fin - fin_inferencia) * 1000}
        return resultado

    def track(self, frame, classes=None, conf=0.25, imgsz=NATIVO_IMGSZ, tracker=None, quantize=None, **_):
        """Equivalente a model.track(..., persist=True) de Ultralytics. quantize=16 -> inferencia en FP16 (GPU)."""
        if imgsz != NATIVO_IMGSZ and not self._avisado:
            print(f"AVISO: el RT-DETR nativo trabaja siempre a {NATIVO_IMGSZ}; se ignora imgsz={imgsz}")
            self._avisado = True
        cajas, confianzas, clases = self.predecir(frame, conf, classes, fp16=quantize == 16)
        ids = None
        if tracker:
            detecciones = sv.Detections(xyxy=cajas.reshape(-1, 4), confidence=confianzas, class_id=clases)
            detecciones = self._tracker(tracker).update_with_detections(detecciones)
            cajas, confianzas, clases, ids = (detecciones.xyxy, detecciones.confidence,
                                              detecciones.class_id, detecciones.tracker_id)
        return [SimpleNamespace(boxes=_CajasNativas(cajas, confianzas, clases, ids), names=self.names,
                                speed=dict(self.speed))]

    def _tracker(self, ruta_yaml):
        """sv.ByteTrack con los valores del yaml de Ultralytics: track_high_thresh, track_buffer y match_thresh."""
        if ruta_yaml not in self.trackers:
            with open(ruta_yaml, encoding="utf-8") as archivo:
                opciones = yaml.safe_load(archivo)
            self.trackers[ruta_yaml] = make_tracker(
                buffer_frames=opciones["track_buffer"], fps_video=self.fps,
                min_conf=opciones["track_high_thresh"], match_thresh=opciones["match_thresh"])
        return self.trackers[ruta_yaml]


# ---------------- Evaluación (una fila más de la comparativa de la sección 3) ----------------
def epocas_entrenadas_nativo():
    """Épocas que llegó a entrenar (una línea por época en log.txt); si no hay log, las configuradas."""
    log = CARPETA_NATIVO / "log.txt"
    if log.exists():
        return sum(1 for linea in log.read_text(encoding="utf-8").splitlines() if linea.strip())
    return NATIVO_EPOCAS


def gflops_nativo(modelo):
    """GFLOPs con thop, como get_flops de Ultralytics; 0.0 si no se pueden medir."""
    try:
        import thop
        copia = copy.deepcopy(modelo)
        entrada = torch.zeros((1, 3, NATIVO_IMGSZ, NATIVO_IMGSZ), device=next(copia.parameters()).device)
        return thop.profile(copia, inputs=(entrada,), verbose=False)[0] / 1e9 * 2
    except Exception:
        return 0.0


def exportar_pesos_ligeros(ruta):
    """Solo los pesos EMA en FP16 (lo que guarda Ultralytics en best.pt), para comparar tamaños en igualdad."""
    destino = Path(ruta).with_name(f"{Path(ruta).stem}_ligero.pth")
    if not destino.exists() or destino.stat().st_mtime < Path(ruta).stat().st_mtime:
        estado = torch.load(ruta, map_location="cpu", weights_only=False)
        pesos = estado["ema"]["module"] if "ema" in estado else estado["model"]
        torch.save({"model": {clave: valor.half() if valor.is_floating_point() else valor
                              for clave, valor in pesos.items()}}, destino)
    return destino


def mosaico_nativo(imagenes, cajas_por_imagen, ruta_salida, confianzas_por_imagen=None, lado=4, celda=480):
    """Mosaico de lado x lado imágenes con sus cajas (como val_batch0_labels/pred.jpg de Ultralytics)."""
    mosaico = np.full((lado * celda, lado * celda, 3), 114, dtype=np.uint8)
    for indice, ruta_imagen in enumerate(imagenes[:lado * lado]):
        imagen = cv2.imread(str(ruta_imagen))
        escala = celda / max(imagen.shape[:2])
        imagen = cv2.resize(imagen, None, fx=escala, fy=escala)
        for numero, caja in enumerate(cajas_por_imagen[indice]):
            x1, y1, x2, y2 = (np.asarray(caja) * escala).astype(int)
            cv2.rectangle(imagen, (x1, y1), (x2, y2), colors(0, True), 2)
            texto = "plate" if confianzas_por_imagen is None else f"plate {confianzas_por_imagen[indice][numero]:.2f}"
            cv2.putText(imagen, texto, (x1, max(y1 - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, colors(0, True), 1)
        fila, columna = divmod(indice, lado)
        mosaico[fila * celda:fila * celda + imagen.shape[0], columna * celda:columna * celda + imagen.shape[1]] = imagen
        cv2.rectangle(mosaico, (columna * celda, fila * celda), ((columna + 1) * celda, (fila + 1) * celda), (255, 255, 255), 2)
        cv2.putText(mosaico, Path(ruta_imagen).name[:40], (columna * celda + 5, (fila + 1) * celda - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 220), 1)
    cv2.imwrite(str(ruta_salida), mosaico)


def evaluar_rtdetr_nativo(nombre, ruta, imagenes):
    """Misma fila que evaluar_detector() (sección 3) para el RT-DETR nativo de matrículas. Las métricas usan las
    mismas funciones que model.val() de Ultralytics: emparejado por IoU 0.50:0.95 (match_predictions) y ap_per_class.
    Guarda las mismas gráficas que model.val() (curvas PR, F1, P y R y matrices de confusión) en
    results/comparativa_detectores/val_<nombre>/."""
    if IMGSZ_COMPARATIVA != NATIVO_IMGSZ:
        print(f"AVISO: {nombre} trabaja siempre a {NATIVO_IMGSZ} y la comparativa usa imgsz={IMGSZ_COMPARATIVA}")
    modelo = RTDETRNativo(ruta)
    metricas = DetMetrics(names=modelo.names)
    confusion = ConfusionMatrix(names=modelo.names)
    ejemplos_reales, ejemplos_cajas, ejemplos_confianzas = [], [], []     # primeras 16 imágenes, para los mosaicos
    carpeta_graficas = CARPETA_COMPARATIVA / ("val_" + "".join(car if car.isalnum() or car in "-_." else "_"
                                                               for car in nombre).strip("_"))
    carpeta_graficas.mkdir(parents=True, exist_ok=True)
    emparejador = SimpleNamespace(iouv=torch.linspace(0.5, 0.95, 10))     # lo único que usa match_predictions
    for ruta_imagen in imagenes:
        imagen = cv2.imread(str(ruta_imagen))
        cajas, confianzas, clases = modelo.predecir(imagen, conf=0.001)     # conf por defecto de model.val()
        reales = etiquetas_yolo(ruta_imagen, imagen.shape[1], imagen.shape[0])
        clases_reales = np.zeros(len(reales))
        if len(reales) and len(cajas):
            iou = box_iou(torch.from_numpy(reales).float(), torch.from_numpy(cajas).float())
            aciertos = BaseValidator.match_predictions(emparejador, torch.from_numpy(clases).float(),
                                                       torch.from_numpy(clases_reales).float(), iou).numpy()
        else:
            aciertos = np.zeros((len(cajas), 10), dtype=bool)
        metricas.update_stats({"tp": aciertos, "conf": confianzas, "pred_cls": clases.astype(float),
                               "target_cls": clases_reales, "target_img": np.unique(clases_reales),
                               "im_name": Path(ruta_imagen).name})
        confusion.process_batch({"bboxes": torch.from_numpy(cajas).float(), "conf": torch.from_numpy(confianzas),
                                 "cls": torch.from_numpy(clases).float()},
                                {"bboxes": torch.from_numpy(reales).float(), "cls": torch.from_numpy(clases_reales)},
                                conf=0.001)                         # misma conf que model.val()
        if len(ejemplos_reales) < 16:
            visibles = confianzas >= 0.25                                 # umbral de dibujo de Ultralytics
            ejemplos_reales.append(reales)
            ejemplos_cajas.append(cajas[visibles])
            ejemplos_confianzas.append(confianzas[visibles])
    metricas.process(save_dir=carpeta_graficas, plot=True)         # BoxPR_curve.png, BoxF1_curve.png, BoxP/R_curve.png
    for normalizar in (True, False):                                 # confusion_matrix(_normalized).png
        confusion.plot(normalize=normalizar, save_dir=carpeta_graficas)
    mosaico_nativo(imagenes, ejemplos_reales, carpeta_graficas / "val_batch0_labels.jpg")
    mosaico_nativo(imagenes, ejemplos_cajas, carpeta_graficas / "val_batch0_pred.jpg", ejemplos_confianzas)
    print(f"Gráficas de validación de {nombre}: {carpeta_graficas}")
    confianzas_curva, f1 = metricas.curves_results[1][0], metricas.curves_results[1][1][0]   # curva F1-confianza
    conf_optima = float(confianzas_curva[np.argmax(f1)])

    primera = cv2.imread(str(imagenes[0]))
    for _ in range(N_CALENTAMIENTO):
        modelo.predecir(primera, conf=conf_optima)
    etapas, totales = [], []
    for ruta_imagen in imagenes:
        imagen = cv2.imread(str(ruta_imagen))          # la lectura de disco queda fuera de la medida
        _, milisegundos = cronometrar(modelo.predecir, imagen, conf=conf_optima)
        etapas.append(dict(modelo.speed))
        totales.append(milisegundos)
    etapas = pd.DataFrame(etapas)

    return {"Modelo": nombre, "Precision": metricas.box.mp, "Recall": metricas.box.mr,
            "F1 max": float(f1.max()), "mAP50": metricas.box.map50, "mAP50-95": metricas.box.map,
            "Conf. F1 max": conf_optima,
            "Preproceso (ms)": etapas["preprocess"].mean(), "Inferencia (ms)": etapas["inference"].mean(),
            "Postproceso (ms)": etapas["postprocess"].mean(), "Total (ms)": np.mean(totales),
            "Total std (ms)": np.std(totales), "FPS": 1000 / np.mean(totales),
            "Parámetros (M)": sum(parametro.numel() for parametro in modelo.modelo.parameters()) / 1e6,
            "GFLOPs": gflops_nativo(modelo.modelo),
            "Tamaño (MB)": exportar_pesos_ligeros(ruta).stat().st_size / 1e6, "Épocas": epocas_entrenadas_nativo()}


print(f"RT-DETR nativo listo ({NATIVO_FAMILIA}/{NATIVO_TAMANO}, {DISPOSITIVO_NATIVO}) | matrículas: "
      f"{RUTA_RTDETR_NATIVO_MATRICULAS if Path(RUTA_RTDETR_NATIVO_MATRICULAS).exists() else 'sin entrenar todavía'}")

RT-DETR nativo listo (v2/l, cuda) | matrículas: sin entrenar todavía


Anotaciones COCO del mismo reparto train/val de la sección 1 (el repo solo lee COCO). No mueve imágenes ni etiquetas: solo crea `datos/coco_train.json` y `datos/coco_val.json`.

In [11]:
# Anotaciones COCO para el repo a partir de train.txt / val.txt y datos/labels (solo hace falta una vez)
def crear_coco_json(split, salida):
    """Mismas imágenes y cajas que ve Ultralytics en ese split; las rutas apuntan a datos/images sin moverlas."""
    carpeta = CARPETA_IMAGENES.resolve()
    imagenes, anotaciones = [], []
    for id_imagen, ruta in enumerate(imagenes_del_split(DATA, split)):
        ancho, alto = tamano_imagen(ruta)
        imagenes.append({"id": id_imagen, "file_name": Path(os.path.relpath(ruta, carpeta)).as_posix(),
                         "width": ancho, "height": alto})
        for x1, y1, x2, y2 in etiquetas_yolo(ruta, ancho, alto):
            anotaciones.append({"id": len(anotaciones) + 1, "image_id": id_imagen, "category_id": 0,
                                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                                "area": float((x2 - x1) * (y2 - y1)), "iscrowd": 0})
    salida.write_text(json.dumps({"images": imagenes, "annotations": anotaciones,
                                  "categories": [{"id": 0, "name": "plate"}]}), encoding="utf-8")
    print(f"{salida}: {len(imagenes)} imágenes, {len(anotaciones)} matrículas")
    return len(imagenes)


crear_coco_json("train", COCO_TRAIN)
crear_coco_json("val", COCO_VAL)

datos\coco_train.json: 1581 imágenes, 3584 matrículas
datos\coco_val.json: 395 imágenes, 927 matrículas


395

Entrenamiento de matrículas con el solver del repo, dentro del cuaderno y sin workers (en Windows los workers necesitarían un script aparte). Parte de los pesos COCO; el mejor checkpoint según mAP50-95 de validación queda en `runs/rtdetr_nativo/best.pth`.

In [12]:
# Entrenamiento del RT-DETR nativo para matrículas (solo una vez). Necesita las anotaciones COCO de la celda anterior.
def entrenar_rtdetr_nativo():
    n_train = len(json.loads(COCO_TRAIN.read_text(encoding="utf-8"))["images"])
    escribir_config_nativo(n_train)                    # runs/rtdetr_nativo/rtdetrv2_matriculas.yml
    asegurar_pesos_coco()
    dist_utils.setup_seed(NATIVO_SEMILLA)
    load_config.__defaults__[0].clear()                # load_config acumula configs en un dict por defecto
    cfg = YAMLConfig(str(NATIVO_CONFIG_MATRICULAS),
                     tuning=str(NATIVO_PESOS_COCO),    # parte de COCO; la cabeza de 80 clases se descarta
                     seed=NATIVO_SEMILLA)
    solver = TASKS[cfg.yaml_cfg["task"]](cfg)
    solver.fit()
    print(f"Mejor modelo (mAP50-95 de validación): {RUTA_RTDETR_NATIVO_MATRICULAS}")


entrenar_rtdetr_nativo()

Start training
tuning checkpoint from weights\rtdetrv2_r50vd_6x_coco_ema.pth
Load model.state_dict, {'missed': [], 'unmatched': ['decoder.denoising_class_embed.weight', 'decoder.enc_score_head.weight', 'decoder.enc_score_head.bias', 'decoder.dec_score_head.0.weight', 'decoder.dec_score_head.0.bias', 'decoder.dec_score_head.1.weight', 'decoder.dec_score_head.1.bias', 'decoder.dec_score_head.2.weight', 'decoder.dec_score_head.2.bias', 'decoder.dec_score_head.3.weight', 'decoder.dec_score_head.3.bias', 'decoder.dec_score_head.4.weight', 'decoder.dec_score_head.4.bias', 'decoder.dec_score_head.5.weight', 'decoder.dec_score_head.5.bias']}
Initial lr: [1e-05, 0.0001, 0.0001]
building train_dataloader with batch_size=4...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
building val_dataloader with batch_size=4...
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
number of trainable parameters: 42700515


OutOfMemoryError: CUDA out of memory. Tried to allocate 26.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 22.14 GiB is allocated by PyTorch, and 52.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 3. Comparativa YOLO vs RT-DETR

Mismas imágenes de validación, misma resolución y mismo equipo. La columna *Conf. F1 max* es el umbral recomendado para `conf_matricula` en el pipeline. Si se ejecutó la sección 2b y existe su `best.pth`, entra también el RT-DETR nativo, con sus gráficas de validación (curvas PR/F1/P/R y matriz de confusión) en `results/comparativa_detectores/val_<modelo>/`.

In [ ]:
# ---------------- Comparativa YOLO vs RT-DETR (detección de matrículas) ----------------
# Mismas imágenes de validación, misma resolución y mismo equipo para los dos modelos.
# Resultados en results/comparativa_detectores/: comparativa.md (tablas), metricas.png y tiempos.png.
MODELOS_COMPARATIVA = {"YOLO11n": (YOLO, RUTA_YOLO_MATRICULAS), "RT-DETR-l": (RTDETR, RUTA_RTDETR_MATRICULAS)}
COLORES_DETECTORES = ["#1f77b4", "#d62728", "#2ca02c"]
IMGSZ_COMPARATIVA = 640           # la de entrenamiento de los dos
N_CALENTAMIENTO = 10
CARPETA_COMPARATIVA = CARPETA_RESULTADOS / "comparativa_detectores"
CARPETA_COMPARATIVA.mkdir(parents=True, exist_ok=True)


def evaluar_detector(nombre, clase, ruta, imagenes):
    """Métricas de model.val() y tiempo por imagen (batch 1) de un detector de matrículas."""
    modelo = clase(ruta)
    args = args_entrenamiento(ruta)
    if args.get("imgsz") != IMGSZ_COMPARATIVA:
        print(f"AVISO: {nombre} se entrenó a imgsz={args.get('imgsz')} y se evalúa a {IMGSZ_COMPARATIVA}")
    metricas = modelo.val(data=DATA, imgsz=IMGSZ_COMPARATIVA, batch=8, device=DEVICE, verbose=False,
                          project=str(CARPETA_COMPARATIVA.resolve()), name=f"val_{nombre}", exist_ok=True)
    confianzas, f1 = metricas.curves_results[1][0], metricas.curves_results[1][1][0]   # curva F1-confianza
    conf_optima = float(confianzas[np.argmax(f1)])

    opciones = {"imgsz": IMGSZ_COMPARATIVA, "conf": conf_optima, "device": DEVICE, "verbose": False}
    primera = cv2.imread(str(imagenes[0]))
    for _ in range(N_CALENTAMIENTO):
        modelo.predict(primera, **opciones)
    etapas, totales = [], []
    for ruta_imagen in imagenes:
        imagen = cv2.imread(str(ruta_imagen))          # la lectura de disco queda fuera de la medida
        resultado, milisegundos = cronometrar(modelo.predict, imagen, **opciones)
        etapas.append(resultado[0].speed)
        totales.append(milisegundos)
    etapas = pd.DataFrame(etapas)

    return {"Modelo": nombre, "Precision": metricas.box.mp, "Recall": metricas.box.mr,
            "F1 max": float(f1.max()), "mAP50": metricas.box.map50, "mAP50-95": metricas.box.map,
            "Conf. F1 max": conf_optima,
            "Preproceso (ms)": etapas["preprocess"].mean(), "Inferencia (ms)": etapas["inference"].mean(),
            "Postproceso (ms)": etapas["postprocess"].mean(), "Total (ms)": np.mean(totales),
            "Total std (ms)": np.std(totales), "FPS": 1000 / np.mean(totales),
            "Parámetros (M)": sum(parametro.numel() for parametro in modelo.model.parameters()) / 1e6,
            "GFLOPs": get_flops(modelo.model, IMGSZ_COMPARATIVA),
            "Tamaño (MB)": Path(ruta).stat().st_size / 1e6, "Épocas": args.get("epochs")}


imagenes_val = imagenes_del_split(DATA, "val")
filas_detectores = [evaluar_detector(nombre, clase, ruta, imagenes_val)
                    for nombre, (clase, ruta) in MODELOS_COMPARATIVA.items()]
if "evaluar_rtdetr_nativo" in globals() and Path(RUTA_RTDETR_NATIVO_MATRICULAS).exists():   # sección 2b
    filas_detectores.append(evaluar_rtdetr_nativo("RT-DETRv2-L (nativo)", RUTA_RTDETR_NATIVO_MATRICULAS, imagenes_val))
tabla_detectores = pd.DataFrame(filas_detectores).set_index("Modelo")
tabla_detectores.round(4).to_csv(CARPETA_COMPARATIVA / "comparativa.csv")

columnas_metricas = ["Precision", "Recall", "F1 max", "mAP50", "mAP50-95"]
columnas_tiempo = ["Preproceso (ms)", "Inferencia (ms)", "Postproceso (ms)", "Total (ms)"]
markdown = "\n\n".join([
    f"Validación: {len(imagenes_val)} imágenes, imgsz={IMGSZ_COMPARATIVA}, batch 1 para los tiempos. "
    "Equipo: " + " | ".join(f"{clave}: {valor}" for clave, valor in EQUIPO.items()),
    "**Detección**\n\n" + tabla_markdown(tabla_detectores, columnas_metricas + ["Conf. F1 max"]),
    "**Velocidad y tamaño**\n\n" + tabla_markdown(
        tabla_detectores, columnas_tiempo + ["FPS", "Parámetros (M)", "GFLOPs", "Tamaño (MB)"], 1),
])
(CARPETA_COMPARATIVA / "comparativa.md").write_text(markdown, encoding="utf-8")
print(markdown)

grafica_barras(tabla_detectores, columnas_metricas, COLORES_DETECTORES, "Detección de matrículas",
               CARPETA_COMPARATIVA / "metricas.png")
grafica_barras(tabla_detectores, columnas_tiempo, COLORES_DETECTORES, f"Tiempo por imagen ({EQUIPO['GPU']})",
               CARPETA_COMPARATIVA / "tiempos.png", formato="{:.1f}", eje_y="ms")
print("Usa la 'Conf. F1 max' de cada modelo como 'conf_matricula' en DETECTORES (sección 6).")


### Curvas de entrenamiento

Evolución por época de los tres entrenamientos (`results.csv` de Ultralytics y `log.txt` del repo oficial) y `results.png` del RT-DETR nativo, equivalente al que Ultralytics guarda para los otros dos. Solo necesita las secciones 0 y la celda de configuración (no carga modelos ni usa la GPU), así que se puede ejecutar en otro kernel mientras se entrena.

El mAP por época del nativo lo calcula el repo con COCOeval (pycocotools) y el de Ultralytics su propio validador, así que las curvas muestran la tendencia; los valores finales comparables son los de la tabla de arriba.

In [ ]:
# ---------------- Curvas de entrenamiento de los tres modelos ----------------
# Resultados en results/comparativa_detectores/curvas_entrenamiento.png y runs/rtdetr_nativo/results.png
CARPETA_COMPARATIVA = CARPETA_RESULTADOS / "comparativa_detectores"
CARPETA_COMPARATIVA.mkdir(parents=True, exist_ok=True)
LOG_RTDETR_NATIVO = Path("runs/rtdetr_nativo/log.txt")
# Índices de test_coco_eval_bbox en log.txt (orden de COCOeval.stats)
COCO_MAP, COCO_MAP50, COCO_MAP75, COCO_AR100 = 0, 1, 2, 8


def historial_ultralytics(ruta_pesos):
    """results.csv del entrenamiento de Ultralytics (carpeta del run de .../weights/best.pt)."""
    ruta = Path(ruta_pesos).parent.parent / "results.csv"
    if not ruta.exists():
        print(f"No existe {ruta}")
        return None
    tabla = pd.read_csv(ruta)
    tabla.columns = tabla.columns.str.strip()
    return pd.DataFrame({"epoca": tabla["epoch"], "mAP50": tabla["metrics/mAP50(B)"],
                         "mAP50-95": tabla["metrics/mAP50-95(B)"]})


def historial_nativo(ruta=LOG_RTDETR_NATIVO):
    """log.txt del repo oficial: una línea JSON por época con las pérdidas de train y COCOeval de validación."""
    if not ruta.exists():
        print(f"No existe {ruta}")
        return None
    filas = []
    for linea in ruta.read_text(encoding="utf-8").splitlines():
        if not linea.strip():
            continue
        registro = json.loads(linea)
        coco = registro.get("test_coco_eval_bbox", [np.nan] * 12)
        filas.append({"epoca": registro["epoch"] + 1,                 # el repo cuenta desde 0
                      "pérdida total": registro["train_loss"], "pérdida clase (VFL)": registro["train_loss_vfl"],
                      "pérdida caja (L1)": registro["train_loss_bbox"], "pérdida caja (GIoU)": registro["train_loss_giou"],
                      "lr": registro["train_lr"], "mAP50": coco[COCO_MAP50], "mAP50-95": coco[COCO_MAP],
                      "mAP75": coco[COCO_MAP75], "AR100": coco[COCO_AR100]})
    return pd.DataFrame(filas)


def grafica_resultados_nativo(historial, ruta_salida):
    """Equivalente al results.png de Ultralytics para el RT-DETR nativo."""
    columnas = ["pérdida total", "pérdida clase (VFL)", "pérdida caja (L1)", "pérdida caja (GIoU)",
                "lr", "mAP50", "mAP50-95", "AR100"]
    figura, ejes = plt.subplots(2, 4, figsize=(18, 8))
    for eje, columna in zip(ejes.flat, columnas):
        eje.plot(historial["epoca"], historial[columna], marker=".", color="#2ca02c", label="valores")
        if len(historial) >= 5 and columna != "lr":
            suavizado = historial[columna].rolling(5, center=True, min_periods=1).mean()
            eje.plot(historial["epoca"], suavizado, ":", color="black", label="suavizado")
        eje.set_title(columna)
        eje.set_xlabel("época")
        eje.grid(alpha=0.3)
    ejes.flat[0].legend()
    figura.suptitle("RT-DETRv2-L (nativo): entrenamiento de matrículas")
    figura.tight_layout()
    figura.savefig(ruta_salida, dpi=150)
    plt.show()


historiales = {"YOLO11n": historial_ultralytics(RUTA_YOLO_MATRICULAS),
               "RT-DETR-l": historial_ultralytics(RUTA_RTDETR_MATRICULAS),
               "RT-DETRv2-L (nativo)": historial_nativo()}
colores = {"YOLO11n": "#1f77b4", "RT-DETR-l": "#d62728", "RT-DETRv2-L (nativo)": "#2ca02c"}

figura, ejes = plt.subplots(1, 2, figsize=(15, 5))
for eje, metrica in zip(ejes, ["mAP50", "mAP50-95"]):
    for nombre, historial in historiales.items():
        if historial is not None and len(historial):
            mejor = historial[metrica].max()
            eje.plot(historial["epoca"], historial[metrica], color=colores[nombre], label=f"{nombre} (máx. {mejor:.3f})")
    eje.set_title(f"{metrica} de validación por época")
    eje.set_xlabel("época")
    eje.set_ylabel(metrica)
    eje.set_ylim(0, 1)
    eje.grid(alpha=0.3)
    eje.legend()
figura.tight_layout()
figura.savefig(CARPETA_COMPARATIVA / "curvas_entrenamiento.png", dpi=150)
plt.show()

if historiales["RT-DETRv2-L (nativo)"] is not None and len(historiales["RT-DETRv2-L (nativo)"]):
    nativo = historiales["RT-DETRv2-L (nativo)"]
    grafica_resultados_nativo(nativo, LOG_RTDETR_NATIVO.parent / "results.png")
    mejor = nativo.loc[nativo["mAP50-95"].idxmax()]
    print(f"RT-DETR nativo: {len(nativo)} épocas registradas | mejor mAP50-95 (COCOeval) {mejor['mAP50-95']:.3f} "
          f"en la época {int(mejor['epoca'])} (la de best.pth)")

## 4. OCR de matrículas

Preprocesado del recorte, lectura con Tesseract o PaddleOCR-VL, corrección con el formato de las matrículas españolas y `LectorMatriculas`, que vota las lecturas de cada matrícula a lo largo del vídeo.

In [ ]:
# ---------------- OCR de matrículas: Tesseract (clásico) y PaddleOCR-VL (VLM) ----------------
# leer_recortes(ocr, recortes) lee una lista de recortes; LectorMatriculas lo usa en el vídeo con tracking.
# Dependencias: tesseract -> pip install pytesseract (+ programa Tesseract)
#               paddle    -> pip install "transformers>=5.0.0" (+ torch y pillow; mejor con GPU)
# ---------------- Configuración ----------------
MARGEN = 0.08                 # margen extra alrededor de la caja del detector, por lado
ALTO_MIN_RECORTE = 64         # los recortes más bajos se amplían hasta esta altura
ALTO_PROCESADO = 120          # altura de trabajo para el preprocesado
LONGITUD_MIN, LONGITUD_MAX = 4, 10   # longitudes válidas de una matrícula no española
SOLO_FORMATO_ESPANOL = False  # True: descarta todo lo que no tenga formato español

# Ruta al ejecutable de Tesseract (se ignora si no existe, p. ej. si ya está en el PATH)
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESSERACT_CONFIG = ("--oem 3 --psm 7 "
                    "-c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
HILOS_TESSERACT = max(2, (os.cpu_count() or 4) - 1)

# PaddleOCR-VL (0.9B): se usa su tarea de reconocimiento de texto ("OCR:")
PADDLE_MODELO = "PaddlePaddle/PaddleOCR-VL-1.5"
PADDLE_PROMPT = "OCR:"
PADDLE_ALTO = 96              # altura a la que se amplía la matrícula antes de pasarla
PADDLE_MAX_TOKENS = 24        # de sobra para una matrícula

_cache = {}                   # modelos y pools: se crean una sola vez


# ---------------- Recorte ----------------
def recortar(frame, caja, margen=MARGEN):
    """Copia del trozo del frame con la matrícula (más un margen), o None si queda vacío."""
    caja = ampliar_caja(caja, margen, frame.shape[1], frame.shape[0])
    if caja is None:
        return None
    x1, y1, x2, y2 = caja
    recorte = frame[y1:y2, x1:x2].copy()
    if recorte.shape[0] < ALTO_MIN_RECORTE:
        recorte = _escalar_a_alto(recorte, ALTO_MIN_RECORTE)
    return recorte


def _escalar_a_alto(imagen, alto):
    escala = alto / imagen.shape[0]
    return cv2.resize(imagen, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)


# ---------------- Formato de matrículas españolas ----------------
# Actual (desde 2000): 4 cifras + 3 consonantes, sin vocales, Ñ ni Q  -> 1234BCD
# Provincial antigua de Canarias: GC/TF + 4 cifras + 1-2 letras        -> GC1234AB
# Provincial más antigua, solo números: GC/TF + hasta 6 cifras          -> GC123456
_CONSONANTES = "BCDFGHJKLMNPRSTVWXYZ"
RE_ACTUAL = re.compile(rf"\d{{4}}[{_CONSONANTES}]{{3}}")
RE_PROVINCIAL = re.compile(r"(GC|TF)\d{4}[A-Z]{1,2}")
RE_PROVINCIAL_NUMERICA = re.compile(r"(GC|TF)\d{4,6}")
RE_PREFIJO_DUDOSO = re.compile(r"([G6][C0O]|[T7][FE])([A-Z0-9]{4,8})$")

# Confusiones típicas: en la parte numérica, letras -> cifras; en la de letras, al revés
_A_CIFRA = str.maketrans("OQDILZSBGT", "0001125867")
_A_LETRA = str.maketrans("025867", "DZSBGT")


def _formato_espanol(texto):
    """Busca una matrícula española dentro de `texto` (tolera la 'E' europea y el ruido)."""
    for patron in (RE_ACTUAL, RE_PROVINCIAL, RE_PROVINCIAL_NUMERICA):
        coincidencia = patron.search(texto)
        if coincidencia:
            return coincidencia.group(0)
    return _corregir_provincial(texto) or _corregir_actual(texto)


def _corregir_provincial(texto):
    """GC/TF mal leído (6C, 7F...) seguido de la parte numérica y, quizá, letras."""
    coincidencia = RE_PREFIJO_DUDOSO.search(texto)
    if not coincidencia:
        return None
    prefijo = "GC" if coincidencia.group(1)[0] in "G6" else "TF"
    resto = coincidencia.group(2)
    if len(resto) in (5, 6):                                   # GC1234A / GC1234AB
        candidata = prefijo + resto[:4].translate(_A_CIFRA) + resto[4:].translate(_A_LETRA)
        if RE_PROVINCIAL.fullmatch(candidata):
            return candidata
    candidata = prefijo + resto.translate(_A_CIFRA)            # GC123456
    return candidata if RE_PROVINCIAL_NUMERICA.fullmatch(candidata) else None


def _corregir_actual(texto):
    """Prueba ventanas de 7 caracteres corrigiendo confusiones según la posición."""
    for inicio in range(len(texto) - 6):
        ventana = texto[inicio:inicio + 7]
        candidata = ventana[:4].translate(_A_CIFRA) + ventana[4:].translate(_A_LETRA)
        if RE_ACTUAL.fullmatch(candidata):
            return candidata
    return None


def alfanumerico(texto):
    """Mayúsculas y solo letras y números: '1234 bcd' y '1234-BCD' quedan igual."""
    return re.sub(r"[^A-Z0-9]", "", (texto or "").upper())


def _es_espanola(texto):
    return _formato_espanol(texto) == texto


def limpiar(texto):
    """Normaliza una lectura del OCR; None si no parece una matrícula."""
    texto = alfanumerico(texto)
    if not texto:
        return None
    espanola = _formato_espanol(texto)
    if espanola:
        return espanola
    if SOLO_FORMATO_ESPANOL:
        return None
    return texto if LONGITUD_MIN <= len(texto) <= LONGITUD_MAX else None


# ---------------- Preprocesado ----------------
_CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(6, 6))   # 6x6: lo que mejor dio en las pruebas


def _quitar_franja_azul(recorte):
    """Recorta la franja azul europea (con la E) del lado izquierdo, si la hay."""
    hsv = cv2.cvtColor(recorte, cv2.COLOR_BGR2HSV)
    azul = cv2.inRange(hsv, (95, 80, 40), (135, 255, 255))
    tercio_izquierdo = azul[:, : recorte.shape[1] // 3]
    fraccion_azul = tercio_izquierdo.mean(axis=0) / 255
    columnas_azules = np.flatnonzero(fraccion_azul > 0.25)
    return recorte[:, columnas_azules[-1] + 2:] if columnas_azules.size else recorte


def _preparar_gris(recorte):
    """Gris, a altura fija, con contraste mejorado y algo suavizado."""
    gris = cv2.cvtColor(_quitar_franja_azul(recorte), cv2.COLOR_BGR2GRAY)
    gris = _CLAHE.apply(_escalar_a_alto(gris, ALTO_PROCESADO))
    return cv2.GaussianBlur(gris, (3, 3), 0)


def _recortar_placa(gris):
    """Localiza el rectángulo claro de la matrícula (sin rejilla ni margen) y lo devuelve."""
    _, claro = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    lado_nucleo = max(3, gris.shape[0] // 4)       # cierra los huecos de los caracteres
    nucleo = cv2.getStructuringElement(cv2.MORPH_RECT, (lado_nucleo, lado_nucleo))
    claro = cv2.morphologyEx(claro, cv2.MORPH_CLOSE, nucleo)
    _, _, estadisticas, _ = cv2.connectedComponentsWithStats(claro)
    alto, ancho = gris.shape
    candidatas = [componente for componente in estadisticas[1:]
                  if componente[2] > 0.4 * ancho and componente[3] > 0.3 * alto
                  and componente[2] > 1.5 * componente[3]]
    if not candidatas:
        return gris
    izquierda, arriba, ancho_placa, alto_placa, _ = max(candidatas,
                                                         key=lambda componente: componente[4])
    return gris[arriba:arriba + alto_placa, izquierda:izquierda + ancho_placa]


def _solo_caracteres(gris):
    """Deja solo las manchas con forma de carácter, negro sobre blanco. Elimina marco,
    tornillos y rejilla, que son lo que más confunde al OCR. None si no encuentra nada."""
    placa = _recortar_placa(gris)
    if placa.shape[0] < 20:
        return None
    placa = _escalar_a_alto(placa, ALTO_PROCESADO)
    _, tinta = cv2.threshold(placa, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    alto, ancho = tinta.shape
    cantidad, etiquetas, estadisticas, _ = cv2.connectedComponentsWithStats(tinta)
    caracteres = np.zeros_like(tinta)
    for indice in range(1, cantidad):
        _, _, ancho_mancha, alto_mancha, area = estadisticas[indice]
        parece_caracter = (0.3 * alto <= alto_mancha <= 0.98 * alto
                           and 0.02 * ancho <= ancho_mancha <= 0.2 * ancho
                           and alto_mancha > 0.9 * ancho_mancha
                           and area >= 0.15 * ancho_mancha * alto_mancha)
        if parece_caracter:
            caracteres[etiquetas == indice] = 255
    return cv2.bitwise_not(caracteres) if caracteres.any() else None


def _con_borde_blanco(imagen, grosor=15):
    return cv2.copyMakeBorder(imagen, grosor, grosor, grosor, grosor,
                              cv2.BORDER_CONSTANT, value=255)


# ---------------- Tesseract ----------------
def _variantes_tesseract(recorte):
    """Varias preparaciones del recorte: Tesseract acierta con unas u otras según la imagen."""
    gris = _preparar_gris(recorte)
    _, otsu = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptativa = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 31, 10)
    variantes = [otsu, gris, adaptativa]
    caracteres = _solo_caracteres(gris)
    if caracteres is not None:
        variantes.insert(0, caracteres)
    return [_con_borde_blanco(variante) for variante in variantes]


def _mejor_lectura(lecturas):
    """La lectura con formato español más repetida; si no hay, la más larga."""
    espanolas = [_formato_espanol(alfanumerico(lectura)) for lectura in lecturas]
    espanolas = [lectura for lectura in espanolas if lectura]
    if espanolas:
        return Counter(espanolas).most_common(1)[0][0]
    return max(lecturas, key=len, default="")


def _ocr_tesseract(recortes):
    import pytesseract

    if TESSERACT_CMD and os.path.exists(TESSERACT_CMD):
        pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
    pool = _cache.setdefault("pool_tesseract", ThreadPoolExecutor(HILOS_TESSERACT))

    # Cada llamada a Tesseract es un proceso aparte: se lanzan todas en paralelo
    tareas = [(indice, pool.submit(pytesseract.image_to_string, variante, config=TESSERACT_CONFIG))
              for indice, recorte in enumerate(recortes)
              for variante in _variantes_tesseract(recorte)]
    lecturas = defaultdict(list)
    for indice, tarea in tareas:
        texto = tarea.result().strip()
        if texto:
            lecturas[indice].append(texto)
    return [_mejor_lectura(lecturas[indice]) for indice in range(len(recortes))]


# ---------------- PaddleOCR-VL ----------------
def _cargar_paddle():
    if "paddle" not in _cache:
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
        tipo = torch.bfloat16 if dispositivo == "cuda" else torch.float32
        procesador = AutoProcessor.from_pretrained(PADDLE_MODELO)
        modelo = AutoModelForImageTextToText.from_pretrained(PADDLE_MODELO, dtype=tipo)
        _cache["paddle"] = (procesador, modelo.to(dispositivo).eval(), dispositivo)
    return _cache["paddle"]


def _ocr_paddle(recortes):
    import torch
    from PIL import Image

    procesador, modelo, dispositivo = _cargar_paddle()
    textos = []
    for recorte in recortes:                     # una matrícula cada vez, como en su ejemplo
        # El modelo está entrenado con fotos reales: se le pasa la matrícula en color,
        # sin la franja azul y ampliada para que los caracteres tengan un tamaño cómodo
        imagen = _escalar_a_alto(_quitar_franja_azul(recorte), PADDLE_ALTO)
        imagen = Image.fromarray(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
        mensajes = [{"role": "user", "content": [{"type": "image", "image": imagen},
                                                 {"type": "text", "text": PADDLE_PROMPT}]}]
        entradas = procesador.apply_chat_template(mensajes, add_generation_prompt=True,
                                                  tokenize=True, return_dict=True,
                                                  return_tensors="pt").to(dispositivo)
        with torch.inference_mode():
            salida = modelo.generate(**entradas, max_new_tokens=PADDLE_MAX_TOKENS, do_sample=False)
        generados = salida[0, entradas["input_ids"].shape[1]:]
        textos.append(procesador.decode(generados, skip_special_tokens=True))
    return textos


# ---------------- Lectura ----------------
_OCRS = {"tesseract": _ocr_tesseract, "paddle": _ocr_paddle}


def _validar_ocr(nombre):
    clave = nombre.strip().lower()
    if clave not in _OCRS:
        raise ValueError(f"OCR desconocido: {nombre!r}. Opciones: {list(_OCRS)}")
    return clave


def leer_recortes(ocr, recortes):
    """Lee una lista de recortes y devuelve (lecturas limpias o None, errores).
    Si el lote falla, se reintenta de uno en uno para que un recorte problemático no anule a los demás."""
    try:
        return [limpiar(bruto) for bruto in _OCRS[ocr](recortes)], [False] * len(recortes)
    except ImportError:                           # problema de instalación: parar y avisar
        raise
    except Exception:
        pass
    lecturas, errores = [], []
    for recorte in recortes:
        try:
            lecturas.append(limpiar(_OCRS[ocr]([recorte])[0])); errores.append(False)
        except ImportError:
            raise
        except Exception as error:                # un fallo puntual no debe tumbar el vídeo
            print(f"[{ocr}] error en una matrícula: {type(error).__name__}: {error}")
            lecturas.append(None); errores.append(True)
    return lecturas, errores


class LectorMatriculas:
    """
    Lee las matrículas de un vídeo con tracking sin frenar el bucle principal:
      - El OCR se ejecuta en segundo plano.
      - Cada ID se lee como mucho cada `cada_n` frames, y solo si se ve al menos tan grande
        como en su mejor lectura anterior (`margen_tamano`).
      - Las lecturas de un ID se votan y cada voto pesa según el área de la matrícula:
        una lectura a 120 px vale mucho más que una a 50 px.
      - Una lectura con `votos_fijar` repeticiones que duplica en peso a la siguiente
        queda fijada, y ese ID deja de pasar por el OCR.
      - Las matrículas más estrechas que `ancho_min` px no se leen.
    """

    def __init__(self, ocr, cada_n=3, votos_fijar=3, max_intentos=25, ancho_min=40,
                 margen_tamano=0.85):
        self.ocr = _validar_ocr(ocr)
        self.cada_n = cada_n
        self.votos_fijar = votos_fijar
        self.max_intentos = max_intentos
        self.ancho_min = ancho_min
        self.margen_tamano = margen_tamano

        self.pesos = defaultdict(Counter)       # id -> lectura -> peso acumulado
        self.repeticiones = defaultdict(Counter)  # id -> lectura -> nº de lecturas
        self.fijadas = {}                       # id -> lectura definitiva
        self.intentos = Counter()
        self.ultimo_frame = {}
        self.mayor_ancho = {}                   # id -> mayor ancho con el que se ha leído
        self.numero_frame = 0
        self._pendiente = None
        self._hilo = ThreadPoolExecutor(max_workers=1)

    # ---------- uso desde el bucle del vídeo ----------
    def actualizar(self, frame, matriculas):
        """Lanza el OCR de las matrículas que lo necesiten y rellena m["texto"]
        con la mejor lectura hasta ahora."""
        self.numero_frame += 1
        self._recoger_resultados()
        if self._pendiente is None:              # solo un lote de OCR a la vez
            self._lanzar_lote(frame, matriculas)
        for matricula in matriculas:
            if matricula.get("id") is not None:
                matricula["texto"] = self.texto(matricula["id"]) or ""

    def texto(self, id_matricula):
        """Mejor lectura actual de un ID (o None)."""
        if id_matricula in self.fijadas:
            return self.fijadas[id_matricula]
        ranking = self._ranking(id_matricula)
        return ranking[0][0] if ranking else None

    def resumen(self):
        """{id: (lectura, nº de lecturas iguales, fijada)}."""
        return {id_matricula: (self.texto(id_matricula),
                               self.repeticiones[id_matricula][self.texto(id_matricula)],
                               id_matricula in self.fijadas)
                for id_matricula in self.pesos}

    def cerrar(self):
        """Espera al último lote de OCR y libera el hilo."""
        self._recoger_resultados(esperar=True)
        self._hilo.shutdown(wait=True)

    # ---------- decisión de qué leer ----------
    def necesita_lectura(self, id_matricula, ancho):
        if id_matricula is None or id_matricula in self.fijadas:
            return False
        frames_desde_ultima = self.numero_frame - self.ultimo_frame.get(id_matricula, -10**9)
        return (self.intentos[id_matricula] < self.max_intentos
                and frames_desde_ultima >= self.cada_n
                and ancho >= self.ancho_min
                and ancho >= self.margen_tamano * self.mayor_ancho.get(id_matricula, 0))

    def marcar_lectura(self, id_matricula, ancho):
        self.intentos[id_matricula] += 1
        self.ultimo_frame[id_matricula] = self.numero_frame
        self.mayor_ancho[id_matricula] = max(ancho, self.mayor_ancho.get(id_matricula, 0))

    def _lanzar_lote(self, frame, matriculas):
        lote = []                                # (id, recorte, ancho)
        ids_en_lote = set()
        for matricula in matriculas:
            id_matricula, caja = matricula.get("id"), matricula["box"]
            ancho = float(caja[2] - caja[0])
            if id_matricula in ids_en_lote or not self.necesita_lectura(id_matricula, ancho):
                continue
            recorte = recortar(frame, caja)
            if recorte is None:
                continue
            self.marcar_lectura(id_matricula, ancho)
            ids_en_lote.add(id_matricula)
            lote.append((id_matricula, recorte, ancho))
        if lote:
            self._pendiente = self._hilo.submit(self._leer_lote, lote)

    def _leer_lote(self, lote):
        textos, _ = leer_recortes(self.ocr, [recorte for _, recorte, _ in lote])
        return [(id_matricula, texto, ancho)
                for (id_matricula, _, ancho), texto in zip(lote, textos, strict=True)]

    def _recoger_resultados(self, esperar=False):
        if self._pendiente is None or not (esperar or self._pendiente.done()):
            return
        resultados = self._pendiente.result()
        self._pendiente = None
        for id_matricula, texto, ancho in resultados:
            self.registrar(id_matricula, texto, ancho)

    # ---------- votación ----------
    def registrar(self, id_matricula, texto, ancho):
        """Añade una lectura (ya limpia) de una matrícula vista con `ancho` píxeles."""
        if not texto:
            return
        self.pesos[id_matricula][texto] += (ancho / 60) ** 2     # peso ~ área
        self.repeticiones[id_matricula][texto] += 1
        ranking = self._ranking(id_matricula)
        mejor, peso_mejor = ranking[0]
        peso_segunda = ranking[1][1] if len(ranking) > 1 else 0
        if (self.repeticiones[id_matricula][mejor] >= self.votos_fijar
                and peso_mejor >= 2 * peso_segunda and _es_espanola(mejor)):
            self.fijadas[id_matricula] = mejor

    def _ranking(self, id_matricula):
        """Lecturas ordenadas: primero las de formato español, luego por peso."""
        return sorted(self.pesos.get(id_matricula, {}).items(),   # .get: no crea entradas vacías
                      key=lambda lectura_peso: (_es_espanola(lectura_peso[0]), lectura_peso[1]),
                      reverse=True)


## 5. Extras del vídeo

Flujo de personas y vehículos por dirección de salida, y anonimización tapando personas y matrículas con `mandril.jpg`.

In [ ]:
# ---------------- Extras del vídeo: flujo por direcciones y anonimización ----------------
# El pipeline (sección 6) los usa si CONTAR_FLUJO / ANONIMIZAR están a True.
FLUJO_FRAMES_PERDIDO = 25        # un id que lleva estos frames sin verse se da por salido (1 s a 25 fps)
FLUJO_MARGEN_BORDE = 0.08        # fracción del frame que cuenta como "borde" de salida
FLUJO_DESPLAZAMIENTO_MIN = 0.10  # si no sale por un borde: desplazamiento mínimo (fracción del frame)
FLECHAS = {"izquierda": "<-", "derecha": "->", "arriba": "^", "abajo": "v"}

IMAGEN_ANONIMIZAR = cv2.imread("mandril.jpg")
CLASES_ANONIMIZAR = {"person"}   # además se tapan todas las matrículas
MARGEN_ANONIMIZAR = 0.05         # se amplía un poco la caja para que no asome nada por los bordes
if IMAGEN_ANONIMIZAR is None:
    raise FileNotFoundError("No encuentro mandril.jpg junto al cuaderno")


class ContadorFlujo:
    """Cuenta por qué lado sale cada persona / vehículo: izquierda, derecha, arriba o abajo.
    Cuando un id lleva FLUJO_FRAMES_PERDIDO frames sin verse se decide su salida:
      1) si su último centro estaba cerca de un borde, sale por ese borde;
      2) si se perdió en medio de la imagen, por la dirección dominante de su desplazamiento;
      3) si apenas se movió, "sin determinar".
    Los ids que siguen en imagen al acabar el vídeo se cierran en filas_resumen()."""

    def __init__(self, ancho, alto):
        self.ancho, self.alto = ancho, alto
        self.primera = {}                     # id -> centro la primera vez que se ve
        self.ultima = {}                      # id -> (numero_frame, centro) la última vez
        self.votos_clase = defaultdict(Counter)
        self.salidas = Counter()              # (clase, direccion) -> total
        self.cerrados = set()

    def actualizar(self, objetos, numero_frame):
        for objeto in objetos:
            id_objeto = objeto["id"]
            if id_objeto is None or id_objeto in self.cerrados:
                continue
            self.primera.setdefault(id_objeto, np.array(centro(objeto["box"])))
            self.ultima[id_objeto] = (numero_frame, np.array(centro(objeto["box"])))
            self.votos_clase[id_objeto][objeto["nombre"]] += 1
        for id_objeto, (visto, _) in list(self.ultima.items()):
            if numero_frame - visto >= FLUJO_FRAMES_PERDIDO:
                self._cerrar(id_objeto)

    def _direccion(self, id_objeto):
        centro_x, centro_y = self.ultima[id_objeto][1]
        distancias = {"izquierda": centro_x / self.ancho, "derecha": 1 - centro_x / self.ancho,
                      "arriba": centro_y / self.alto, "abajo": 1 - centro_y / self.alto}
        borde = min(distancias, key=distancias.get)
        if distancias[borde] <= FLUJO_MARGEN_BORDE:
            return borde
        dx, dy = (self.ultima[id_objeto][1] - self.primera[id_objeto]) / np.array([self.ancho, self.alto])
        if max(abs(dx), abs(dy)) < FLUJO_DESPLAZAMIENTO_MIN:
            return "sin determinar"
        if abs(dx) >= abs(dy):
            return "derecha" if dx > 0 else "izquierda"
        return "abajo" if dy > 0 else "arriba"

    def _cerrar(self, id_objeto):
        clase = self.votos_clase[id_objeto].most_common(1)[0][0]
        self.salidas[(clase, self._direccion(id_objeto))] += 1
        self.cerrados.add(id_objeto)
        del self.ultima[id_objeto]

    def dibujar(self, frame):
        lineas = [f"{clase} {FLECHAS.get(direccion, '?')} {direccion}: {total}"
                  for (clase, direccion), total in sorted(self.salidas.items())]
        dibujar_panel(frame, ["Salidas"] + lineas if lineas else [], derecha=True)

    def filas_resumen(self):
        """Filas para el CSV de flujo: [clase, direccion, total]."""
        for id_objeto in list(self.ultima):
            self._cerrar(id_objeto)
        return [[clase, direccion, total] for (clase, direccion), total in sorted(self.salidas.items())]


def _tapar(frame, caja):
    """Pega el mandril sobre la caja, escalado para cubrirla sin deformarse (recorta lo que sobra)."""
    caja = ampliar_caja(caja, MARGEN_ANONIMIZAR, frame.shape[1], frame.shape[0])
    if caja is None:
        return
    x1, y1, x2, y2 = caja
    ancho, alto = x2 - x1, y2 - y1
    escala = max(ancho / IMAGEN_ANONIMIZAR.shape[1], alto / IMAGEN_ANONIMIZAR.shape[0])
    nuevo_ancho = max(ancho, round(IMAGEN_ANONIMIZAR.shape[1] * escala))
    nuevo_alto = max(alto, round(IMAGEN_ANONIMIZAR.shape[0] * escala))
    escalada = cv2.resize(IMAGEN_ANONIMIZAR, (nuevo_ancho, nuevo_alto),
                          interpolation=cv2.INTER_AREA if escala < 1 else cv2.INTER_LINEAR)
    desplazamiento_x, desplazamiento_y = (nuevo_ancho - ancho) // 2, (nuevo_alto - alto) // 2
    frame[y1:y2, x1:x2] = escalada[desplazamiento_y:desplazamiento_y + alto,
                                   desplazamiento_x:desplazamiento_x + ancho]


def anonimizar(frame, objetos, matriculas):
    """Tapa con el mandril a las personas y todas las matrículas. Recibe una copia del frame sin dibujar."""
    for caja in [objeto["box"] for objeto in objetos if objeto["nombre"] in CLASES_ANONIMIZAR] + \
                [matricula["box"] for matricula in matriculas]:
        _tapar(frame, caja)
    return frame


## 6. Pipeline de vídeo

Un único pipeline para los detectores (YOLO, RT-DETR de Ultralytics y RT-DETR nativo de la sección 2b). Genera en `results/` el vídeo anotado, el CSV de detecciones (con `texto_matricula`), los totales por clase, las matrículas leídas, el flujo y el vídeo anonimizado.

In [ ]:
# ---------------- Pipeline de vídeo: detección, tracking, OCR y extras ----------------
# main(detector="yolo" | "rtdetr" | "rtdetr_nativo", ocr="paddle" | "tesseract" | None). Necesita las secciones 0, 4 y 5.
# ---------------- Configuración ----------------
# Cada detector con sus modelos y resoluciones. RT-DETR debe ejecutarse a la resolución de
# entrenamiento (640): a 1280 apenas detectaba matrículas.
DETECTORES = {
    "yolo": {"clase": YOLO, "general": "yolo11s.pt", "matriculas": RUTA_YOLO_MATRICULAS,
             "imgsz_general": 960, "imgsz_matriculas": 1280, "conf_matricula": 0.4},
    "rtdetr": {"clase": RTDETR, "general": "rtdetr-l.pt", "matriculas": RUTA_RTDETR_MATRICULAS,
               "imgsz_general": 640, "imgsz_matriculas": IMGSZ_RTDETR,
               "conf_matricula": 0.4},   # pon la "Conf. F1 max" de RT-DETR que da la comparativa (3)
}
if "RTDETRNativo" in globals():   # RT-DETR sin Ultralytics (sección 2b)
    DETECTORES["rtdetr_nativo"] = {"clase": RTDETRNativo, "general": NATIVO_COCO,
                                   "matriculas": RUTA_RTDETR_NATIVO_MATRICULAS,
                                   "imgsz_general": NATIVO_IMGSZ, "imgsz_matriculas": NATIVO_IMGSZ,
                                   "conf_matricula": 0.4}   # pon la "Conf. F1 max" del nativo que da la comparativa (3)

OCR_CADA_N = 3                 # cada matrícula se lee como mucho cada N frames
OCR_VOTOS_FIJAR = 3            # lecturas iguales para dar una matrícula por buena
OCR_ANCHO_MIN = 40             # px: las matrículas más estrechas no se leen

# Índices COCO: person=0, bicycle=1, car=2, motorcycle=3, bus=5, truck=7
CLASES = [0, 1, 2, 3, 5, 7]
VEHICULOS_CON_MATRICULA = {2, 3, 5, 7}

CONF_TRACKER_GENERAL = 0.1     # el tracker recibe también las detecciones dudosas
CONF_TRACKER_MATRICULAS = 0.05
CONF_DIBUJO_GENERAL = 0.3      # confianza mínima para dibujar y guardar

MIN_FRAMES = 5                 # frames que debe verse un track para recibir ID visible
COLOR_MATRICULA = (0, 255, 255)

MOSTRAR = True                 # False: sin ventana (más rápido)
VENTANA = "Tracking + matriculas"
TAM_VENTANA = (1280, 720)

CONTAR_FLUJO = True            # flujo por direcciones (sección 5)
ANONIMIZAR = True              # vídeo adicional con personas y matrículas tapadas (sección 5)

OPCIONES_TRACKING = {"persist": True, "verbose": False, "device": DEVICE,
                     **({"quantize": 16} if torch.cuda.is_available() else {})}   # FP16 en GPU

CABECERA_CSV = ["fotograma", "tipo_objeto", "confianza", "identificador_tracking",
                "x1", "y1", "x2", "y2",
                "matricula_en_su_caso", "confianza_matricula",
                "mx1", "my1", "mx2", "my2", "texto_matricula"]


# ---------------- Utilidades ----------------
class RenumeradorIds:
    """Da IDs consecutivos (1, 2, 3...) a los tracks que se han visto al menos `min_frames`."""

    def __init__(self, min_frames):
        self.min_frames = min_frames
        self.veces_visto = defaultdict(int)
        self.ids_visibles = {}
        self._siguiente = count(1)

    def __call__(self, id_tracker):
        if id_tracker is None:
            return None
        self.veces_visto[id_tracker] += 1
        if self.veces_visto[id_tracker] < self.min_frames:
            return None
        if id_tracker not in self.ids_visibles:
            self.ids_visibles[id_tracker] = next(self._siguiente)
        return self.ids_visibles[id_tracker]

    @property
    def total(self):
        return len(self.ids_visibles)


class LectorVideo:
    """Decodifica fotogramas en un hilo aparte para que no frene la detección."""

    def __init__(self, captura, tam_cola=8):
        self.captura = captura
        self.cola = queue.Queue(tam_cola)
        self.parar = False
        self.hilo = threading.Thread(target=self._leer, daemon=True)
        self.hilo.start()

    def _leer(self):
        while not self.parar:
            ok, frame = self.captura.read()
            self.cola.put(frame if ok else None)
            if not ok:
                break

    def leer(self):
        return self.cola.get()        # None al terminar el vídeo

    def detener(self):
        self.parar = True
        try:                          # vaciar la cola para desbloquear el hilo
            while True:
                self.cola.get_nowait()
        except queue.Empty:
            pass
        self.hilo.join(timeout=2)


def rutas_salida(detector, ocr):
    base = str(CARPETA_RESULTADOS / f"{detector}_{ocr or 'sin_ocr'}")
    rutas = {"video": f"{base}.mp4", "detecciones": f"{base}_detecciones.csv",
             "totales": f"{base}_totales.csv"}
    if ocr:
        rutas["matriculas"] = f"{base}_matriculas.csv"
    if CONTAR_FLUJO:
        rutas["flujo"] = f"{base}_flujo.csv"
    if ANONIMIZAR:
        rutas["anonimo"] = f"{base}_anonimo.mp4"
    return rutas


def detecciones(resultado):
    """Lista de (caja, confianza, clase, id_tracker) de un resultado de YOLO o RT-DETR."""
    cajas = resultado.boxes
    ids = cajas.id.int().cpu().tolist() if cajas.id is not None else [None] * len(cajas)
    return zip(cajas.xyxy.cpu().numpy(), cajas.conf.cpu().tolist(),
               cajas.cls.int().cpu().tolist(), ids, strict=True)


# ---------------- Procesado de un frame ----------------
def recoger_objetos(resultado, nombres_clases, renumerador, votos_clase):
    objetos = []
    for caja, confianza, clase, id_tracker in detecciones(resultado):
        if confianza < CONF_DIBUJO_GENERAL:
            continue
        id_visible = renumerador(id_tracker)
        nombre = nombres_clases[clase]
        if id_visible is not None:
            votos_clase[id_visible][nombre] += 1
        objetos.append({"box": caja, "conf": confianza, "cls": clase, "nombre": nombre,
                        "id": id_visible, "matricula": None})
    return objetos


def vehiculo_que_contiene(caja, vehiculos):
    centro_x, centro_y = centro(caja)
    return next((vehiculo for vehiculo in vehiculos
                 if vehiculo["box"][0] <= centro_x <= vehiculo["box"][2]
                 and vehiculo["box"][1] <= centro_y <= vehiculo["box"][3]), None)


def recoger_matriculas(resultado, objetos, renumerador, conf_min):
    """Asocia cada matrícula al vehículo que la contiene; devuelve las que no tienen vehículo."""
    vehiculos = [objeto for objeto in objetos if objeto["cls"] in VEHICULOS_CON_MATRICULA]
    sueltas = []
    for caja, confianza, _, id_tracker in detecciones(resultado):
        if confianza < conf_min:
            continue
        matricula = {"box": caja, "conf": confianza, "id": renumerador(id_tracker), "texto": ""}
        vehiculo = vehiculo_que_contiene(caja, vehiculos)
        if vehiculo is None:
            sueltas.append(matricula)
        elif vehiculo["matricula"] is None or confianza > vehiculo["matricula"]["conf"]:
            vehiculo["matricula"] = matricula     # si hay varias, la más fiable
    return sueltas


def totales_por_clase(votos_clase, numero_matriculas):
    """Objetos únicos por clase (cada ID cuenta una vez, con su clase mayoritaria)."""
    totales = Counter(votos.most_common(1)[0][0] for votos in votos_clase.values())
    totales["matricula"] = numero_matriculas
    return totales


# ---------------- Dibujo y CSV ----------------
def dibujar(imagen, caja, texto, color):
    x1, y1, x2, y2 = map(int, caja)
    cv2.rectangle(imagen, (x1, y1), (x2, y2), color, 2)
    (ancho_texto, alto_texto), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    y_texto = max(y1, alto_texto + 6)
    cv2.rectangle(imagen, (x1, y_texto - alto_texto - 6), (x1 + ancho_texto + 4, y_texto), color, -1)
    cv2.putText(imagen, texto, (x1 + 2, y_texto - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)


def etiqueta_matricula(matricula):
    if matricula["texto"]:
        return matricula["texto"]
    return f"plate #{matricula['id']}" if matricula["id"] is not None else "plate"


def columnas_matricula(matricula):
    if matricula is None:
        return [""] * 7
    x1, y1, x2, y2 = map(int, matricula["box"])
    id_matricula = matricula["id"] if matricula["id"] is not None else ""
    return [id_matricula, f"{matricula['conf']:.3f}", x1, y1, x2, y2, matricula["texto"]]


def anotar_frame(frame, numero_frame, objetos, sueltas, totales):
    """Dibuja sobre el frame y devuelve las filas del CSV de ese frame."""
    filas = []
    for objeto in objetos:
        etiqueta = f"{objeto['nombre']} #{objeto['id']}" if objeto["id"] is not None else objeto["nombre"]
        dibujar(frame, objeto["box"], etiqueta, colors(objeto["cls"], True))
        matricula = objeto["matricula"]
        if matricula is not None:
            dibujar(frame, matricula["box"], etiqueta_matricula(matricula), COLOR_MATRICULA)
        x1, y1, x2, y2 = map(int, objeto["box"])
        filas.append([numero_frame, objeto["nombre"], f"{objeto['conf']:.3f}",
                      objeto["id"] if objeto["id"] is not None else "",
                      x1, y1, x2, y2, *columnas_matricula(matricula)])
    for matricula in sueltas:
        dibujar(frame, matricula["box"], etiqueta_matricula(matricula), COLOR_MATRICULA)
        filas.append([numero_frame, "matricula", "", "", "", "", "", "", *columnas_matricula(matricula)])
    dibujar_panel(frame, [f"{clase}: {numero}" for clase, numero in sorted(totales.items())])
    return filas


# ---------------- Programa principal ----------------
def main(detector="yolo", ocr="paddle"):
    """detector: "yolo", "rtdetr" o "rtdetr_nativo" | ocr: "paddle", "tesseract" o None (sin lectura de matrículas)."""
    cfg = DETECTORES[detector]
    rutas = rutas_salida(detector, ocr)
    modelo_general = cfg["clase"](cfg["general"])
    modelo_matriculas = cfg["clase"](cfg["matriculas"])
    if detector == "rtdetr":
        imgsz_entreno = args_entrenamiento(cfg["matriculas"]).get("imgsz")
        if imgsz_entreno != cfg["imgsz_matriculas"]:
            print(f"AVISO: el modelo de matrículas se entrenó a {imgsz_entreno} y se ejecuta a {cfg['imgsz_matriculas']}")
    lector_ocr = (LectorMatriculas(ocr, cada_n=OCR_CADA_N, votos_fijar=OCR_VOTOS_FIJAR,
                                   ancho_min=OCR_ANCHO_MIN) if ocr else None)

    captura = cv2.VideoCapture(VIDEO)
    fps = captura.get(cv2.CAP_PROP_FPS) or 30
    ancho = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    for modelo in (modelo_general, modelo_matriculas):
        if hasattr(modelo, "configurar_video"):   # RT-DETR nativo: su tracker (sv.ByteTrack) necesita los fps
            modelo.configurar_video(fps)
    dispositivo = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Vídeo: {ancho}x{alto} @ {fps:.1f} fps  |  dispositivo: {dispositivo}  |  "
          f"detector: {detector}  |  OCR: {ocr}")

    escritor_video = cv2.VideoWriter(rutas["video"], cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))
    archivo_csv = open(rutas["detecciones"], "w", newline="", encoding="utf-8")
    escritor_csv = csv.writer(archivo_csv)
    escritor_csv.writerow(CABECERA_CSV)
    escritor_anonimo = (cv2.VideoWriter(rutas["anonimo"], cv2.VideoWriter_fourcc(*"mp4v"), fps,
                                        (ancho, alto)) if ANONIMIZAR else None)
    flujo = ContadorFlujo(ancho, alto) if CONTAR_FLUJO else None

    # la vista se reduce antes de mostrarla: pintar un frame grande en la ventana es caro
    escala_vista = min(TAM_VENTANA[0] / ancho, TAM_VENTANA[1] / alto, 1.0)
    tam_vista = (int(ancho * escala_vista), int(alto * escala_vista))
    if MOSTRAR:
        cv2.namedWindow(VENTANA, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(VENTANA, *TAM_VENTANA)

    ids_objetos = RenumeradorIds(MIN_FRAMES)
    ids_matriculas = RenumeradorIds(MIN_FRAMES)
    votos_clase = defaultdict(Counter)       # id visible -> votos de clase

    video = LectorVideo(captura)
    numero_frame = 0
    inicio = time.time()
    try:
        while (frame := video.leer()) is not None:
            numero_frame += 1
            resultado = modelo_general.track(frame, classes=CLASES, conf=CONF_TRACKER_GENERAL,
                                             imgsz=cfg["imgsz_general"], tracker="tracker_general.yaml",
                                             **OPCIONES_TRACKING)[0]
            resultado_matriculas = modelo_matriculas.track(
                frame, conf=CONF_TRACKER_MATRICULAS, imgsz=cfg["imgsz_matriculas"],
                tracker="tracker_matriculas.yaml", **OPCIONES_TRACKING)[0]

            objetos = recoger_objetos(resultado, modelo_general.names, ids_objetos, votos_clase)
            sueltas = recoger_matriculas(resultado_matriculas, objetos, ids_matriculas,
                                         cfg["conf_matricula"])

            # OCR en segundo plano, con votación por ID de matrícula (rellena "texto")
            matriculas = [objeto["matricula"] for objeto in objetos if objeto["matricula"]] + sueltas
            if lector_ocr:
                lector_ocr.actualizar(frame, matriculas)

            # Extras: se calculan sobre el frame limpio, antes de dibujar encima
            if flujo:
                flujo.actualizar(objetos, numero_frame)
            if escritor_anonimo:
                escritor_anonimo.write(anonimizar(frame.copy(), objetos, matriculas))

            totales = totales_por_clase(votos_clase, ids_matriculas.total)
            escritor_csv.writerows(anotar_frame(frame, numero_frame, objetos, sueltas, totales))
            if flujo:
                flujo.dibujar(frame)
            escritor_video.write(frame)

            if numero_frame % 100 == 0:
                velocidad = numero_frame / (time.time() - inicio)
                print(f"  frame {numero_frame}/{total_frames}  ({velocidad:.1f} fps de procesado)")

            if MOSTRAR:
                vista = frame if escala_vista == 1.0 else cv2.resize(frame, tam_vista,
                                                                     interpolation=cv2.INTER_AREA)
                cv2.imshow(VENTANA, vista)
                if (cv2.waitKey(1) & 0xFF == ord("q")
                        or cv2.getWindowProperty(VENTANA, cv2.WND_PROP_VISIBLE) < 1):
                    break
    finally:
        video.detener()
        captura.release()
        escritor_video.release()
        archivo_csv.close()
        if escritor_anonimo:
            escritor_anonimo.release()
        if lector_ocr:
            lector_ocr.cerrar()
        cv2.destroyAllWindows()

    totales = totales_por_clase(votos_clase, ids_matriculas.total)
    guardar_csv(rutas["totales"], ["clase", "total"], sorted(totales.items()))
    resumen = lector_ocr.resumen() if lector_ocr else {}
    if lector_ocr:
        guardar_csv(rutas["matriculas"], ["id_matricula", "texto", "votos", "confirmada"],
                    [[id_matricula, texto, votos, "si" if fijada else "no"]
                     for id_matricula, (texto, votos, fijada) in sorted(resumen.items())])
    if flujo:
        guardar_csv(rutas["flujo"], ["clase", "direccion", "total"], flujo.filas_resumen())

    confirmadas = sum(fijada for _, _, fijada in resumen.values())
    print(f"\nProcesados {numero_frame} frames en {time.time() - inicio:.1f} s")
    print("\nTotal de objetos únicos por clase:")
    for clase, numero in sorted(totales.items()):
        print(f"  {clase}: {numero}")
    if lector_ocr:
        print(f"\nMatrículas leídas: {len(resumen)} ({confirmadas} confirmadas)")
    print("\nGuardado:", ", ".join(rutas.values()))
    return {"totales": totales, "matriculas": resumen, "rutas": rutas}


In [ ]:
# Opciones: detector="yolo" | "rtdetr", ocr="paddle" | "tesseract" | None
resultado_yolo = main(detector="yolo", ocr="paddle")


In [ ]:
# Mismo vídeo con RT-DETR
# resultado_rtdetr = main(detector="rtdetr", ocr="paddle")


In [ ]:
# Mismo vídeo con el RT-DETR sin Ultralytics (necesita la primera celda de la sección 2b)
# resultado_rtdetr_nativo = main(detector="rtdetr_nativo", ocr="paddle")

## 7. Comparativa OCR: clásico vs VLM

Tesseract frente a PaddleOCR-VL sobre las matrículas del dataset con texto real: acierto exacto, CER (error por carácter), tiempo en lote y latencia.

In [ ]:
# ---------------- Comparativa OCR: clásico (Tesseract) vs VLM (PaddleOCR-VL) ----------------
# Mismas matrículas, mismo recorte y preprocesado (los del vídeo) y mismo equipo para los dos OCR.
# El texto real sale del campo "plate" de datos/annotations/*.json (cualquier split: los OCR no se entrenan).
# Se descartan solo las matrículas sin texto real (None, "None" o vacío); el resto de la imagen se usa.
# Resultados en results/comparativa_ocr/: comparativa_ocr.md (tablas), acierto.png y tiempos.png.
OCRS_A_COMPARAR = {"tesseract": "Tesseract (clásico)", "paddle": "PaddleOCR-VL (VLM)"}
COLORES_OCR = ["#1f77b4", "#d62728"]
N_IMAGENES_OCR = None            # nº de imágenes (muestra aleatoria fija); None = todas
SEMILLA_OCR = 42
ANCHO_MIN_OCR = 40               # px: más estrechas no son legibles (igual que OCR_ANCHO_MIN del vídeo)
N_CALENTAMIENTO_OCR = 3          # lecturas previas sin medir (carga del modelo, CUDA...)
LOTE_OCR = 16                    # matrículas por llamada en la medida principal (como en el vídeo)
N_LATENCIA_OCR = 30              # matrículas leídas de una en una para medir la latencia
TRAMOS_ANCHO_OCR = [(40, 70, "40-70 px"), (70, 110, "70-110 px"), (110, 1e9, "> 110 px")]
CARPETA_OCR = CARPETA_RESULTADOS / "comparativa_ocr"
CARPETA_OCR.mkdir(parents=True, exist_ok=True)


def cargar_muestras(n_imagenes=N_IMAGENES_OCR, semilla=SEMILLA_OCR):
    """Matrículas con texto real. Se barajan los nombres de los JSON (sin abrirlos) y se leen hasta reunir
    n_imagenes con alguna matrícula válida. Devuelve un DataFrame: imagen, x1, y1, x2, y2, ancho, texto."""
    rutas_json = sorted(CARPETA_ANOTACIONES.glob("*.json"))
    if n_imagenes is not None:
        rutas_json = [rutas_json[indice] for indice in np.random.default_rng(semilla).permutation(len(rutas_json))]
    filas, imagenes_usadas, descartadas = [], 0, Counter()
    for ruta_json in rutas_json:
        if n_imagenes is not None and imagenes_usadas >= n_imagenes:
            break
        ruta_imagen = CARPETA_IMAGENES / f"{ruta_json.stem}.jpg"
        if not ruta_imagen.exists():
            continue
        anotaciones = json.loads(ruta_json.read_text(encoding="utf-8"))
        ancho_imagen, alto_imagen = tamano_imagen(ruta_imagen)
        filas_imagen = []
        for anotacion in anotaciones:
            texto = alfanumerico(anotacion.get("plate"))
            centro_x, centro_y, ancho, alto = anotacion["rectangle"]
            # caja en píxeles, ajustada a la imagen (una anotación que se sale no debe dar un recorte vacío)
            x1, y1 = max(0, int((centro_x - ancho / 2) * ancho_imagen)), max(0, int((centro_y - alto / 2) * alto_imagen))
            x2, y2 = min(ancho_imagen, int((centro_x + ancho / 2) * ancho_imagen)), min(alto_imagen, int((centro_y + alto / 2) * alto_imagen))
            if texto in ("", "NONE"):
                descartadas["sin texto real"] += 1
            elif x2 - x1 < ANCHO_MIN_OCR or y2 <= y1:
                descartadas[f"menos de {ANCHO_MIN_OCR} px"] += 1
            else:
                filas_imagen.append({"imagen": str(ruta_imagen), "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                                     "ancho": x2 - x1, "texto": texto})
        if filas_imagen:                 # si se descartan todas sus matrículas, la imagen no cuenta
            filas += filas_imagen
            imagenes_usadas += 1
    if not filas:
        raise ValueError("No hay ninguna matrícula válida: revisa CARPETA_ANOTACIONES y ANCHO_MIN_OCR")
    muestras = pd.DataFrame(filas)
    muestras.to_csv(CARPETA_OCR / "muestras.csv", index_label="id")
    print(f"{imagenes_usadas} imágenes -> {len(muestras)} matrículas | descartadas: {dict(descartadas)}")
    return muestras


def preparar_recortes(muestras):
    """Recorta las matrículas leyendo cada imagen una sola vez: en memoria solo quedan los recortes."""
    recortes = [None] * len(muestras)
    for ruta, grupo in muestras.groupby("imagen", sort=False):
        imagen = cv2.imread(ruta)
        if imagen is None:
            print(f"AVISO: no se pudo leer {ruta}; sus matrículas contarán como error")
            continue
        for fila in grupo.itertuples():
            recortes[fila.Index] = recortar(imagen, (fila.x1, fila.y1, fila.x2, fila.y2))
    print(f"{len(recortes)} recortes preparados "
          f"({sum(recorte.nbytes for recorte in recortes if recorte is not None) / 1024 ** 2:.0f} MB)")
    return recortes


def distancia_edicion(texto_a, texto_b):
    """Distancia de Levenshtein: inserciones, borrados y sustituciones para pasar de un texto a otro."""
    anterior = list(range(len(texto_b) + 1))
    for posicion_a, caracter_a in enumerate(texto_a, start=1):
        actual = [posicion_a]
        for posicion_b, caracter_b in enumerate(texto_b, start=1):
            actual.append(min(anterior[posicion_b] + 1, actual[posicion_b - 1] + 1,
                              anterior[posicion_b - 1] + (caracter_a != caracter_b)))
        anterior = actual
    return anterior[-1]


def evaluar_ocr(ocr, muestras, recortes):
    """Lee todas las matrículas en lotes de LOTE_OCR. Tiempo por matrícula = tiempo del lote / tamaño."""
    leer_recortes(ocr, [recorte for recorte in recortes[:N_CALENTAMIENTO_OCR] if recorte is not None])
    filas = []
    for desde in range(0, len(muestras), LOTE_OCR):
        lote = recortes[desde:desde + LOTE_OCR]
        (lecturas, errores), milisegundos = cronometrar(leer_recortes, ocr, lote)
        for fila, leido, error in zip(muestras.iloc[desde:desde + LOTE_OCR].itertuples(), lecturas, errores):
            leido = alfanumerico(leido)
            errores_caracter = distancia_edicion(leido, fila.texto)
            filas.append({"id": fila.Index, "ancho": fila.ancho, "texto_real": fila.texto, "texto_leido": leido,
                          "exacto": leido == fila.texto, "sin_lectura": leido == "", "error": error,
                          "errores": errores_caracter, "cer": errores_caracter / len(fila.texto),
                          "tiempo_ms": milisegundos / len(lote)})
        print(f"\r  {ocr}: {len(filas)}/{len(muestras)} matrículas", end="")
    resultado = pd.DataFrame(filas)
    resultado.to_csv(CARPETA_OCR / f"lecturas_{ocr}.csv", index=False)
    print(f"\n{ocr}: acierto exacto {resultado['exacto'].mean():.1%}, CER {resultado['cer'].mean():.3f}, "
          f"{resultado['tiempo_ms'].mean():.1f} ms/matrícula")
    return resultado


def medir_latencia(ocr, recortes, n_recortes=N_LATENCIA_OCR, semilla=SEMILLA_OCR):
    """Tiempo de leer UNA matrícula suelta, sobre un subconjunto fijo (el mismo para todos los OCR)."""
    validos = [recorte for recorte in recortes if recorte is not None]
    indices = np.random.default_rng(semilla).choice(len(validos), min(n_recortes, len(validos)), replace=False)
    leer_recortes(ocr, [validos[indices[0]]])                     # calentamiento
    tiempos = np.array([cronometrar(leer_recortes, ocr, [validos[indice]])[1] for indice in indices])
    print(f"{ocr}: latencia {tiempos.mean():.1f} ms/matrícula ({len(tiempos)} matrículas)")
    return tiempos


def informe_ocr(resultados, latencias):
    """resultados: {ocr: evaluar_ocr(...)}, latencias: {ocr: medir_latencia(...)}. Tablas y 2 gráficas."""
    filas = []
    for ocr, resultado in resultados.items():
        fila = {"OCR": OCRS_A_COMPARAR[ocr], "Acierto exacto (%)": 100 * resultado["exacto"].mean(),
                "≤ 1 error (%)": 100 * (resultado["errores"] <= 1).mean(), "CER medio": resultado["cer"].mean(),
                "Sin lectura (%)": 100 * resultado["sin_lectura"].mean(),
                "Errores del OCR (%)": 100 * resultado["error"].mean(),
                "En lote (ms)": resultado["tiempo_ms"].mean(), "Latencia (ms)": latencias[ocr].mean(),
                "Latencia std (ms)": latencias[ocr].std(), "Matrículas/s (lote)": 1000 / resultado["tiempo_ms"].mean()}
        for minimo, maximo, tramo in TRAMOS_ANCHO_OCR:
            fila[f"Acierto {tramo} (%)"] = 100 * resultado[resultado["ancho"].between(minimo, maximo, inclusive="left")]["exacto"].mean()
        filas.append(fila)
    tabla = pd.DataFrame(filas).set_index("OCR")
    tabla.round(3).to_csv(CARPETA_OCR / "comparativa_ocr.csv")

    columnas_acierto = ["Acierto exacto (%)", "≤ 1 error (%)", "Sin lectura (%)"]
    columnas_tiempo = ["En lote (ms)", "Latencia (ms)"]
    primero = next(iter(resultados.values()))
    markdown = "\n\n".join([
        f"{len(primero)} matrículas (ancho ≥ {ANCHO_MIN_OCR} px), mismo recorte y preprocesado para los dos OCR. "
        f"*En lote*: lotes de {LOTE_OCR} (como en el vídeo; Tesseract los procesa en paralelo y PaddleOCR-VL de uno en uno). "
        f"*Latencia*: {len(next(iter(latencias.values())))} matrículas leídas de una en una. "
        "Equipo: " + " | ".join(f"{clave}: {valor}" for clave, valor in EQUIPO.items()),
        "**Tasa de acierto**\n\n" + tabla_markdown(tabla, columnas_acierto[:2] + ["CER medio"] + columnas_acierto[2:]
                                                   + ["Errores del OCR (%)"], 2),
        "**Tiempo por matrícula**\n\n" + tabla_markdown(tabla, columnas_tiempo + ["Latencia std (ms)", "Matrículas/s (lote)"], 1),
        "**Acierto exacto según el ancho de la matrícula**\n\n" + tabla_markdown(
            tabla, [f"Acierto {tramo} (%)" for *_, tramo in TRAMOS_ANCHO_OCR], 1),
    ])
    (CARPETA_OCR / "comparativa_ocr.md").write_text(markdown, encoding="utf-8")
    print(markdown)

    grafica_barras(tabla, columnas_acierto, COLORES_OCR, f"Tasa de acierto OCR ({len(primero)} matrículas)",
                   CARPETA_OCR / "acierto.png", formato="{:.1f}%", eje_y="% de matrículas")
    grafica_barras(tabla, columnas_tiempo, COLORES_OCR, f"Tiempo por matrícula ({EQUIPO['GPU']})",
                   CARPETA_OCR / "tiempos.png", formato="{:.1f}", eje_y="ms (escala log)", log=True)
    return tabla


In [ ]:
# Evaluar los dos OCR sobre las mismas matrículas y generar tablas y gráficas
muestras = cargar_muestras()
recortes = preparar_recortes(muestras)                                                     # cada imagen se lee una vez
resultados_ocr = {ocr: evaluar_ocr(ocr, muestras, recortes) for ocr in OCRS_A_COMPARAR}    # acierto + tiempo en lote
latencias_ocr = {ocr: medir_latencia(ocr, recortes) for ocr in OCRS_A_COMPARAR}           # latencia (1 a 1)
tabla_ocr = informe_ocr(resultados_ocr, latencias_ocr)
